In [ ]:
# Carl et al. (in prep.): measurement script
# Copyright (C) 2026 Friedrich Carl
# This file is part of Carl et al. (in prep.).
#
# The code is free software: you can redistribute it and/or modify
# it under the terms of the GNU General Public License as published by
# the Free Software Foundation, either version 3 of the License, or
# (at your option) any later version.
#
# The code is distributed in the hope that it will be useful,
# but WITHOUT ANY WARRANTY; without even the implied warranty of
# MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the
# GNU General Public License for more details.
#
# You should have received a copy of the GNU General Public License
# along with MyProject.  If not, see  https://www.gnu.org/licenses/gpl-3.0.txt.


In [ ]:
"""This is the script to measure the lateral and vertical dimensions as well as gradient data of a threshold model, as
described in Carl et al. (in prep., 2026). For explanations on why certain steps in the code are being executed (that way), please refer to the paper.
Please remember to change all paths/directories in this script according to your system"""

In [ ]:
"""Loading data & creating initial cross sections parallel to longer horizontal axis of bounding box"""
import pyvista as pv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import colormaps
from matplotlib.patches import Polygon
from matplotlib.collections import PatchCollection
from shapely.geometry import LineString
import gempy as gp
import vtk

reader = vtk.vtkPolyDataReader()
reader.SetFileName(r"C:\Users\carl\Desktop\Paper 3\random datasets standard models\Juliusburg_Koestorf-Rosenthal\10 sections 0 boreholes\vtk hulls\10 sections 0 boreholes_hull_GE_150_smoothed.vtk") #read .vtk input
reader.Update()

# Get the data from the VTK file
data = reader.GetOutput()

# Extract points and their associated data
points = data.GetPoints()
num_points = points.GetNumberOfPoints()
point_data = data.GetPointData()

# Create a list to store point coordinates and associated data
point_list = []

# Iterate through points and their data
for i in range(num_points):
    # Get point coordinates
    point = points.GetPoint(i)
    # Append point coordinates to the list
    point_list.append(list(point))

# Create a pandas DataFrame
columns = ['x', 'y', 'z']
df = pd.DataFrame(point_list, columns=columns)


mesh = pv.read(r"C:\Users\carl\Desktop\Paper 3\random datasets standard models\Juliusburg_Koestorf-Rosenthal\10 sections 0 boreholes\vtk hulls\10 sections 0 boreholes_hull_GE_150_smoothed.vtk")

# Create a regular bounding box around the irregular mesh
bounds = mesh.bounds
regular_box = pv.Box(bounds)
extents = mesh.bounds

# Print the extents
print("X Extent:", extents[0], "-", extents[1])
print("Y Extent:", extents[2], "-", extents[3])
print("Z Extent:", extents[4], "-", extents[5])

box_extents = regular_box.bounds

# Get the vertices of the bounding box
box_vertices = regular_box.points

# Calculate the minimum and maximum values of each axis (X, Y, and Z)
xmin = np.min(box_vertices[:, 0])
xmax = np.max(box_vertices[:, 0])
ymin = np.min(box_vertices[:, 1])
ymax = np.max(box_vertices[:, 1])
zmin = np.min(box_vertices[:, 2])
zmax = np.max(box_vertices[:, 2])

dx = xmax - xmin
dy = ymax - ymin
dz = zmax - zmin

#p0 at xmin, midy, midz; p1 at xmax, midy, midz
p0 = [xmin, ymin + dy/2, zmin + dz/2]
p1 = [xmax, ymin + dy/2, zmin + dz/2]


n_cuts = 32
# Automatically choose the plane normal of all cross sections and adjust the center_array calculation
if dx >= dy:
    plane_normal = [1, 0, 0]  # Perpendicular to the X-axis
    print("Plane normal is [1, 0, 0] (perpendicular to the X-axis).")
    # Distribute the centers along the Y axis
    center_array = np.ones((n_cuts, 3))
    center_array[:, 0] = np.linspace(xmin, xmax, n_cuts)  # Linearly spaced values for x-coordinates
    center_array[:, 1] = ymin + dy / 2  # Constant value for y-coordinates (midpoint of Y-axis)
    center_array[:, 2] = zmin + dz / 2  # Constant value for z-coordinates (midpoint of Z-axis)
else:
    plane_normal = [0, 1, 0]  # Perpendicular to the Y-axis
    print("Plane normal is [0, 1, 0] (perpendicular to the Y-axis).")
    # Distribute the centers along the X axis
    center_array = np.ones((n_cuts, 3))
    center_array[:, 0] = xmin + dx / 2  # Constant value for x-coordinates (midpoint of X-axis)
    center_array[:, 1] = np.linspace(ymin, ymax, n_cuts)  # Linearly spaced values for y-coordinates
    center_array[:, 2] = zmin + dz / 2  # Constant value for z-coordinates (midpoint of Z-axis)
center_array

"""-----------------------------------------------------initial parallel cross sections ---------------------------------------------------"""

sliced_meshes = []

# Initialize the plotter
plotter = pv.Plotter()

# Loop through each center in center_array
for i in range(len(center_array)):
    for g in np.arange(0,95,5):
        # Access the i-th entry in center_array
        cross_section_center = center_array[i]

        if dx >= dy:
            plane_normal = [1, 0, 0]  # Perpendicular to the X-axis
        else:
            plane_normal = [0, 1, 0]  # Perpendicular to the Y-axis
    
        # Calculate the origin of the slicing plane
        origin = cross_section_center
    
        # Define the slicing plane for the irregular mesh
        Section_dir1 = mesh.slice(normal=plane_normal, origin=cross_section_center)
        
    
    # Define the slicing plane for the regular bounding box
    bounds = mesh.bounds
    regular_box = pv.Box(bounds)
    boundbox = regular_box.slice(normal=plane_normal, origin=cross_section_center)
    
    # Check if the sliced mesh for the irregular mesh is not empty
    if Section_dir1.n_points > 0:
        # Add the sliced mesh to the plotter
        plotter.add_mesh(Section_dir1, color='red', opacity=0.5, name=f'Irregular Mesh {i}')  # Add the sections for the irregular input mesh
            
        # Append the sliced mesh to the list
        sliced_meshes.append(Section_dir1)
    
    """ in case you want to visualize the bounding box of the input mesh as well: uncomment below"""
    if boundbox.n_points > 0:
        # Add the sliced mesh to the plotter
        plotter.add_mesh(boundbox, color='blue', opacity=0.5, name=f'Regular Bounding Box {i}')  # Add the slice for the regular bounding box
        
# Show the plotter
plotter.show()

In [ ]:
"""compute all rotation steps for all cross sections created before (5° intervals), to find the orientation of every 
cross section perpendicular to the longitudinal horizontal main axis (1st direction)"""
import pyvista as pv
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.transform import Rotation as R
from sklearn.preprocessing import MinMaxScaler

#compute the centroid of a cross section
def centroid(points):
    x = [p[0] for p in points]
    y = [p[1] for p in points]
    centroid = (sum(x) / len(points), sum(y) / len(points))
    return centroid

#sort the points of a section by angle
def sort_points_by_angle(points, centroid):
    def angle(p):
        return np.arctan2(p[1] - centroid[1], p[0] - centroid[0])
    return sorted(points, key=angle)


# Normalize the vertices
def normalize_values(vertices):
    scaler = MinMaxScaler()
    normalized_vertices = scaler.fit_transform(vertices)
    return normalized_vertices


# Function to compute the area using the shoelace formula
def shoelace_area(vertices):
    n = len(vertices)
    area = 0.5 * abs(sum(vertices[i][0]*vertices[(i+1) % n][1] - vertices[i][1]*vertices[(i+1) % n][0] for i in range(n)))
    return area

#function to rotate & project a section parallel to the XY plane / rotate the normal parallel to the x axis
def rotate_to_x_axis(vertices, normal):
    normal = normal / np.linalg.norm(normal)  # Normalize the normal vector
    target = np.array([1, 0, 0])  # Target vector along the x-axis

    # Ensure normal is not aligned with the target vector to avoid zero rotation
    if np.allclose(normal, target) or np.allclose(normal, -target):
        return vertices, np.eye(3)  # No rotation needed if already aligned

    # Calculate the rotation needed to align the normal with the x-axis
    rotation_vector = np.cross(normal, target)
    rotation_vector /= np.linalg.norm(rotation_vector)  # Normalize the rotation vector
    angle = np.arccos(np.clip(np.dot(normal, target), -1.0, 1.0))
    
    K = np.array([
        [0, -rotation_vector[2], rotation_vector[1]],
        [rotation_vector[2], 0, -rotation_vector[0]],
        [-rotation_vector[1], rotation_vector[0], 0]
    ])
    rotation_matrix = np.eye(3) + np.sin(angle) * K + (1 - np.cos(angle)) * np.dot(K, K)

    if np.linalg.det(rotation_matrix) < 0:
        rotation_matrix[:, 1] = -rotation_matrix[:, 1]
        
    # Apply the rotation to the vertices
    rotated_vertices = vertices @ rotation_matrix.T 
    return rotated_vertices, rotation_matrix
    
#compute the center point of a cross section
def compute_center_point(vertices, plane_normal):
    if np.array_equal(plane_normal, [1, 0, 0]):
        # Compute mean in YZ plane, but return full 3D center (x remains the same)
        center = np.mean(vertices[:, 1:], axis=0)  # Mean of Y and Z coordinates
        return np.array([np.mean(vertices[:, 0]), center[0], center[1]])  # Add X back
    elif np.array_equal(plane_normal, [0, 1, 0]):
      #Compute mean in XZ plane, but return full 3D center (y remains the same)
        center = np.mean(vertices[:, [0, 2]], axis=0)  # Mean of X and Z coordinates
        return np.array([center[0], np.mean(vertices[:, 1]), center[1]])  # Add Y back
    else:
        raise ValueError("Unsupported plane normal")

# Initialize the plotter
plotter = pv.Plotter()

# Initialize a dictionary to store area values for each section
area_values = {i: [] for i in range(len(center_array))}

"""---------------------------------------------------compute rotation steps---------------------------------------------------"""
# Loop through each center in center_array
for i in range(len(center_array)):
    
    # Access the i-th entry in center_array
    cross_section_center = center_array[i]
    
    if dx >= dy:
        plane_normal = [1, 0, 0]  # Perpendicular to the X-axis
    else:
        plane_normal = [0, 1, 0]  # Perpendicular to the Y-axis

    # Define the initial slicing plane
    Section_dir1 = mesh.slice(normal=plane_normal, origin=cross_section_center)
    
    if Section_dir1.n_points > 0:
        # Add the initial sliced mesh to the plotter
        plotter.add_mesh(Section_dir1, color='red', opacity=0.5, name=f'Initial Section {i}')
        
        # Extract vertices of the slice mesh
        vertices = Section_dir1.points

        
        center_point = compute_center_point(vertices, plane_normal)
        
        # Initialize list to store rotated normal vectors
        rotated_normals = []

        # Loop to rotate the normal vector stepwise by 5°
        for g in range(38):  # 38 steps will give 0° to 180° rotation
            # Rotation angle in radians
            theta = np.radians(g * 5)
    
            # Rotation matrix about z-axis
            rotation_matrix = np.array([
                [np.cos(theta), np.sin(theta), 0],
                [-np.sin(theta), np.cos(theta), 0],
                [0, 0, 1]
                ])
           
    
            # Compute rotated normal vector
            rotated_n = np.dot(rotation_matrix, plane_normal)
 
            
            # Append rotated normal to list
            rotated_normals.append(rotated_n)
    
        # Loop through rotated normals
        for rotated_n in rotated_normals:

            rotated_section = mesh.slice(normal=rotated_n, origin=center_point)
            
            # Check if the sliced mesh for the irregular mesh is not empty
            if rotated_section.n_points > 0:
                # Add the rotated sliced mesh to the plotter
                plotter.add_mesh(rotated_section, color='blue', opacity=0.5, name=f'Irregular Mesh {i} - Rotation {int(np.degrees(np.arccos(rotated_n[0])))}°')
                
                # Extract vertices of the rotated slice mesh
                rotated_vertices = rotated_section.points

                #project the cross section into the YZ plane for area calculation
                vertices_centered = rotated_vertices - center_point
                rotated_vertices, rotation_matrix = rotate_to_x_axis(vertices_centered, rotated_n)
                rotated_vertices += center_point
                
                # Convert vertices to the required format for area calculation (YZ-plane)
                yz_projection = rotated_vertices[:, 1:3]
                
                
                points = yz_projection

                # Find centroid
                cnt = centroid(points)
                
                # Sort points by angle
                sorted_points = sort_points_by_angle(points, cnt)
                
                # Calculate area
                area = shoelace_area(sorted_points)
                
                # Store the area value
                area_values[i].append(area)

# Show the plotter
plotter.show()

# Access and print the area values for each center
for center_index, areas in area_values.items():
    print(f"Center {center_index}: {areas}")
    print()  # blank line after each center's areas


In [ ]:
"""determining & saving the orientation of 1st direction for every cross section"""
import json

min_area_steps = {}

#determine the rotation showing the minimum area for every cross section
def calculate_min_steps():
    # Loop through each center in area_values dictionary
    for center_index, areas in area_values.items():
        
        # Find the rotation step with the smallest area value 
        min_area_value = float('inf')
        min_area_step = None
        
        for step, area in enumerate(areas):
            if area > 0 and area < min_area_value:
                min_area_value = area
                min_area_step = step
        
        """ Calculate the step value and apply the condition for min_step == 180°, that min_step==180° is instead=0° (both sections are the same
        regarding their dimensions (just mirrored), but this definition avoids a bug) """
        if min_area_step is not None:
            min_step = min_area_step * 5
            if min_step == 180:
                min_step = 0  # Set it to 0 if the step is exactly 180
        else:
            min_step = None
        
        # Store the step corresponding to the minimum area for this center
        min_area_steps[center_index] = min_step

def get_min_steps():
    # Ensure the calculate_min_steps function is called before returning the results
    calculate_min_steps()
    return min_area_steps

# Now call the get_min_steps to populate and retrieve the min_area_steps
min_steps = get_min_steps()

# Save min_steps to a JSON file
with open('min_steps.json', 'w') as f:
    json.dump(min_steps, f)

# Print the rotation step with the smallest area value for each center
for center_index, min_step in min_steps.items():
    if min_step is not None:
        print(f"Smallest area for center {center_index} at rotation step {min_step}°: {area_values[center_index][min_step//5]}")
    else:
        print(f"No area value greater than 0 found for center {center_index}")


In [ ]:
"""visualization of sections and computation of center points"""
import pyvista as pv
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
import json

# Initialize the plotter
plotter = pv.Plotter()

# Create a colormap for different centers
num_centers = len(center_array)
colormap = cm.get_cmap('viridis', num_centers)
center_points = []
center_points_dict = {}

#centers_to_display = [21]

# First pass: compute unique center points for each slice
for idx, cross_section_center in enumerate(center_array):
    # Define the plane normal based on dx and dy
    if dx >= dy:
        plane_normal = np.array([1, 0, 0])  # Perpendicular to the X-axis
    else:
        plane_normal = np.array([0, 1, 0])  # Perpendicular to the Y-axis
    
    # Slice the mesh along the plane
    rotated_slice = mesh.slice(normal=plane_normal, origin=cross_section_center)
    
    # Extract vertices of the rotated slice mesh
    rotated_vertices = rotated_slice.points
    
    if len(rotated_vertices) > 0:
        # Calculate the center point (geometric center) of the vertices
        center_point_i = compute_center_point(rotated_vertices, plane_normal)
        center_points.append(center_point_i)
        center_points_dict[f'center_{idx}'] = center_point_i.tolist()  # Store as list for JSON compatibility
    else:
        # Handle empty slice cases by storing NaNs
        center_points_dict[f'center_{idx}'] = [np.nan, np.nan, np.nan]
        center_points.append([np.nan, np.nan, np.nan]) 

# Save the center points dictionary to a JSON file
with open('center_points.json', 'w') as f:
    json.dump(center_points_dict, f)

print(f"Center points saved in 'center_points.json'.")

center_points = np.array(center_points)
print(f"Computed Center Points: {center_points}")  # Debugging statement

# Iterate through each center in the array for visualization
# Filter out NaN center points and store their valid indices
for idx, center_point in enumerate(center_points):
    # Skip if the center point is NaN (no valid slice)
    #if idx not in centers_to_display:
     #   continue  # Skip this center if it's not in the display list
    
    if np.isnan(center_point).any():
        continue

    # Get the minimal rotation step from the previous computation
    min_step = min_area_steps[idx]  # Use idx from the original center_array
    
    if min_step is None:
        continue  # Skip centers with no valid minimal area step

    # Define the plane normal for slicing again
    if dx >= dy:
        plane_normal = np.array([1, 0, 0])  # Perpendicular to the X-axis
    else:
        plane_normal = np.array([0, 1, 0])  # Perpendicular to the Y-axis
    
    # Rotation angle in radians based on the minimal step
    theta = np.radians(min_step)
    
    # Rotation matrix for the Z-axis rotation
    rotation_matrix = np.array([
        [np.cos(theta), np.sin(theta), 0],
        [-np.sin(theta), np.cos(theta), 0],
        [0, 0, 1]
    ])
    
    # Compute the rotated normal vector
    rotated_n = np.dot(rotation_matrix, plane_normal)
    
    # Define the slicing plane for the rotated normal using the center point
    rotated_slice = mesh.slice(normal=rotated_n, origin=center_point)
    
    # Check if the rotated slice mesh is not empty
    if rotated_slice.n_points > 0:
        # Extract vertices of the rotated slice mesh
        rotated_vertices = rotated_slice.points

        if center_index == 1:  # Corrected variable
            print(f"Rotated vertices for center point index {center_index}:\n", rotated_vertices)
        
        
        # Center the vertices around the computed center point
        vertices_centered = rotated_vertices - center_point
        
        # Optionally rotate the vertices to align with the X-axis
        rotated_vertices, _ = rotate_to_x_axis(vertices_centered, rotated_n)
        
        # Shift the vertices back to the original center point
        rotated_vertices += center_point
        
        # Get color from the colormap
        color = colormap(idx)[:3]  # Get RGB values from the colormap
        
        # Add the rotated slice to the plotter
        plotter.add_mesh(rotated_slice, color=color, line_width=2, label=f'Center {center_index} Slice at {min_step}°')

# Add the original mesh for reference
plotter.add_mesh(mesh, opacity=0.3, color='white')

# Add all center points to the plotter
for center in center_points:
    if not np.isnan(center).any():
        plotter.add_points(center, color='green', point_size=10, render_points_as_spheres=True, label='Cross Section Center')

# Set plot labels and title
plotter.add_axes()

plotter.show_grid(color='white',
    location='outer',
    grid='back',
    ticks='outside',
    font_size=8,
) # Just show the axes first



plotter.set_background('black')
plotter.view_isometric()

# Set the camera position to view along the Z-axis
plotter.view_vector([0, 0, 1])

# Show the plotter
plotter.show()


In [ ]:
"""convert cross sections of 1st direction prior to artifact correction into shapely.linestring objects"""
import pyvista as pv
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.transform import Rotation as R
from sklearn.preprocessing import MinMaxScaler
import matplotlib.cm as cm
from shapely.geometry import LineString
import json

# Load min_steps from the JSON file
with open('min_steps.json', 'r') as f:
    min_steps = json.load(f)
min_steps = {int(k): v for k, v in min_steps.items()}
if 0 not in min_steps:
    min_steps[0] = 0 


def centroid(points):
    if len(points) <= 1:
        raise ValueError("Cannot compute centroid with 1 or fewer points.")
    
    x = [p[0] for p in points]
    y = [p[1] for p in points]
    centroid = (sum(x) / len(points), sum(y) / len(points))
    return centroid

            
def sort_points_by_angle(points, centroid):
    def angle(p):
        return np.arctan2(p[1] - centroid[1], p[0] - centroid[0])
    return sorted(points, key=angle)

# Function to compute the area using the shoelace formula
def shoelace_area(vertices):
    n = len(vertices)
    area = 0.5 * abs(sum(vertices[i][0]*vertices[(i+1) % n][1] - vertices[i][1]*vertices[(i+1) % n][0] for i in range(n)))
    return area

# Normalize the vertices
def normalize_values(vertices):
    scaler = MinMaxScaler()
    normalized_vertices = scaler.fit_transform(vertices)
    return normalized_vertices


# Initialize the plotter
plotter = pv.Plotter()

# Create a colormap for different centers
num_centers = len(center_array)
colormap = cm.get_cmap('viridis', num_centers)

center_points = []
linestrings = {}


# First pass: compute unique center points for each cross section
for idx, cross_section_center in enumerate(center_array):
    if dx >= dy:
        plane_normal = [1, 0, 0]  # Perpendicular to the X-axis
    else:
        plane_normal = [0, 1, 0]  # Perpendicular to the Y-axis

    rotated_section = mesh.slice(normal=plane_normal, origin=cross_section_center)
    
    # Extract vertices of the rotated slice mesh
    initial_vertices = rotated_section.points
    
    # Calculate the center point (geometric center) of the vertices
    center_point_i = compute_center_point(initial_vertices, plane_normal)
    center_points.append(center_point_i)

center_points = np.array(center_points)

# Initialize a dictionary to store area values and minimal steps for each section
min_areas = {i: float('inf') for i in range(len(center_array))}
area_values = {i: [] for i in range(len(center_array))}


# Initialize dictionaries to store the results
min_area_dict = {}
rotated_normals_dict = {}
rotated_norms = []
rotated_norms_dict = {} 

# Populate dictionary after computing areas for each min step
for i in range(len(center_points)):

    if i == 31:
        print(f"Skipping computation for index {i}.")
        continue
    
    if min_area_steps[i] is not None:
        min_area_dict[i] = min_areas[i]  # Store in the min_area_dict

with open('min_area_dict.json', 'w') as f:
    json.dump(min_area_dict, f)

#computing and saving rotated normals for later use    
for i in range(len(center_points)):
    
    # Access the i-th entry in center_points
    center_point = center_points[i]
    
    if dx >= dy:
        plane_normal = [1, 0, 0]  # Perpendicular to the X-axis
    else:
        plane_normal = [0, 1, 0]  # Perpendicular to the Y-axis
    
    # Define the initial slicing plane
    Section_dir1 = mesh.slice(normal=plane_normal, origin=center_point)
    
    if Section_dir1.n_points > 0:
        
        # Extract vertices of the sliced mesh
        vertices = Section_dir1.points
        
        # Initialize list to store rotated normal vectors
        rotated_normals = []

        # Loop to rotate the normal vector stepwise by 5° (up to 180°)
        for g in range(38):  # 38 steps will give 0° to 180° rotation
            # Rotation angle in radians
            theta = np.radians(g * 5)
    
            # Rotation matrix about z-axis
            rotation_matrix = np.array([
                [np.cos(theta), np.sin(theta), 0],
                [-np.sin(theta), np.cos(theta), 0],
                [0, 0, 1]
            ])
           
            # Compute rotated normal vector
            rotated_n = np.dot(rotation_matrix, plane_normal)
            
            # Append rotated normal to list
            rotated_normals.append(rotated_n.tolist())

        rotated_normals_dict[f'rotated_n_{i}'] = rotated_normals

# Save the rotated normals dictionary to a JSON file
with open('rotated_normals.json', 'w') as f:
    json.dump(rotated_normals_dict, f)

#once again computing cross sections to then convert them into shapely.linestring objects
for i in range(len(center_points)):
    if min_area_steps[i] is not None:
        # Get the minimal rotation step and area
        min_step = min_area_steps[i]
        min_area = min_areas[i]
        
        if dx >= dy:
            plane_normal = [1, 0, 0]  # Perpendicular to the X-axis
        else:
            plane_normal = [0, 1, 0]  # Perpendicular to the Y-axis
        
        # Rotation matrix for the minimal step
        theta = np.radians(min_step)
        rotation_matrix = np.array([
            [np.cos(theta), np.sin(theta), 0],
            [-np.sin(theta), np.cos(theta), 0],
            [0, 0, 1]
        ])
        
        # Compute the rotated normal vector
        rotated_norm = np.dot(rotation_matrix, plane_normal)

        rotated_norms.append(rotated_norm.tolist())
        rotated_norms_dict[f'rotated_n_{i}'] = rotated_norm.tolist() 

        
        # Define the slicing plane for the rotated normal using the center point
        rotated_section = mesh.slice(normal=rotated_norm, origin=center_points[i])
        
        # Extract vertices of the rotated sliced mesh
        initial_vertices = rotated_section.points

        

        """Implementation of special case where min_step == 0 / 90; these following special cases are for visualization purposes later on,  if 
        a section is orthogonal to one of the horizontal axes of the initial bounding box"""
        if dx < dy:
            if min_step == 0:
                if len(initial_vertices) > 1:
                    try:
                        # Use the first and third columns (X and Z) when min_step is 0
                        cnt = centroid(initial_vertices[:, [0, 2]])
                        sorted_points = sort_points_by_angle(initial_vertices[:, [0, 2]], cnt)
                        area = shoelace_area(sorted_points)
                    except ValueError as e:
                        print(f"Skipping computation due to error: {e}")
                else:
                    print("Skipping centroid and area calculation due to insufficient points.")
                
                if len(initial_vertices) > 1:
                    line_string = LineString(initial_vertices)
                    linestrings[i] = LineString(initial_vertices)
                else:
                    print(f"Skipping LineString for section {i} due to insufficient points.")
    
                # Append the area and update min_area and min_step
                area_values[i].append(area)
                if area < min_areas[i]:
                    min_areas[i] = area
                    min_steps[i] = min_step

    
            # For min_step != 0, proceeding with the default YZ plane 
            else:
                if len(initial_vertices) > 1:
                    try:
                        cnt = centroid(initial_vertices[:, 1:3])
                        sorted_points = sort_points_by_angle(initial_vertices[:, 1:3], cnt)
                        area = shoelace_area(sorted_points)
                    except ValueError as e:
                        print(f"Skipping computation due to error: {e}")
                else:
                    print("Skipping centroid and area calculation due to insufficient points.")
                
                if len(initial_vertices) > 1:
                    line_string = LineString(initial_vertices)
                    linestrings[i] = LineString(initial_vertices)
                else:
                    print(f"Skipping LineString for section {i} due to insufficient points.")
    
                # Append the area and update min_area and min_step
                area_values[i].append(area)
                if area < min_areas[i]:
                    min_areas[i] = area
                    min_steps[i] = min_step
           
                
        # for dx >= dy, the exception is if the section is rotated by 90°
        elif dx >= dy:
            if min_step == 90:
                if len(initial_vertices) > 1:
                    try:
                        # Use the first and third columns (X and Z) when min_step is 0
                        cnt = centroid(initial_vertices[:, [0, 2]])
                        sorted_points = sort_points_by_angle(initial_vertices[:, [0, 2]], cnt)
                        area = shoelace_area(sorted_points)
                    except ValueError as e:
                        print(f"Skipping computation due to error: {e}")
                else:
                    print("Skipping centroid and area calculation due to insufficient points.")
        
                if len(initial_vertices) > 1:
                    line_string = LineString(initial_vertices)
                    linestrings[i] = LineString(initial_vertices)
                else:
                    print(f"Skipping LineString for section {i} due to insufficient points.")
        
                # Append the area and update min_area and min_step
                area_values[i].append(area)
                if area < min_areas[i]:
                    min_areas[i] = area
                    min_steps[i] = min_step

            # For min_step != 0, proceed with the default YZ plane 
            else:
                if len(initial_vertices) > 1:
                    try:
                        cnt = centroid(initial_vertices[:, 1:3])
                        sorted_points = sort_points_by_angle(initial_vertices[:, 1:3], cnt)
                        area = shoelace_area(sorted_points)
                    except ValueError as e:
                        print(f"Skipping computation due to error: {e}")
                else:
                    print("Skipping centroid and area calculation due to insufficient points.")
                
                if len(initial_vertices) > 1:
                    line_string = LineString(initial_vertices)
                    linestrings[i] = LineString(initial_vertices)
                else:
                    print(f"Skipping LineString for section {i} due to insufficient points.")
    
                # Append the area and update min_area and min_step
                area_values[i].append(area)
                if area < min_areas[i]:
                    min_areas[i] = area
                    min_steps[i] = min_step


# Save the rotated normals dictionary to a JSON file
with open('rotated_norms.json', 'w') as f:
    json.dump(rotated_norms_dict, f)

            

In [ ]:
""" Optional: 2D visualization of cross sections (1st direction) in top view"""
import matplotlib.pyplot as plt
import numpy as np
import pyvista as pv

# Initialize the plot
plt.figure(figsize=(12, 12))

# Create a colormap with 31 distinct colors
colors = plt.cm.tab20c(np.linspace(0, 1, 31))

# Plot all minimal steps (sections of 1st direction) in X-Y plane
for i in range(len(center_points)):
    if min_area_steps[i] is not None:
        # Get the minimal rotation step and area
        min_step = min_area_steps[i]

        if dx >= dy:
            plane_normal = [1, 0, 0]  # Perpendicular to the X-axis
        else:
            plane_normal = [0, 1, 0]  # Perpendicular to the Y-axis

        # Rotation matrix for the minimal step
        theta = np.radians(min_step)
        rotation_matrix = np.array([
            [np.cos(theta), np.sin(theta), 0],
            [-np.sin(theta), np.cos(theta), 0],
            [0, 0, 1]
        ])

        # Compute the rotated normal vector
        rotated_n = np.dot(rotation_matrix, plane_normal)

        # Define the slicing plane for the rotated normal using the center point
        rotated_section = mesh.slice(normal=rotated_n, origin=center_points[i])

        # Extract vertices of the rotated slice mesh
        rotated_vertices = rotated_section.points


        x_rotated = rotated_vertices[:, 0]
        y_rotated = rotated_vertices[:, 1]


        # Plot the 2D projection in X-Y plane (Section_dir1)
        
        plt.plot(x_rotated, y_rotated, 'o', color=colors[i % len(colors)], linestyle='--', label=f'Section {i} - Min Step')


# Set labels and title
plt.xlabel('X')
plt.ylabel('Y')
plt.title('Sections of 1st direction in Top View (X-Y)')

# Add legend
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

# Show the plot
plt.show()


In [ ]:
"""first step of artifact correction: automatic normalized nearest neighbor approach to get rid of most artifacts of vertex order"""
import numpy as np
from shapely.geometry import LineString
from sklearn.preprocessing import MinMaxScaler
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import json

# Load min_steps from the JSON file
with open('min_steps.json', 'r') as f:
    min_steps = json.load(f)
min_steps = {int(k): v for k, v in min_steps.items()}

def get_last_z_value(line_string):
    coords = np.array(line_string.coords)
    return coords[-1, 2]  

def get_z_value(vertices, index):
    return vertices[index][2]  

def normalize_values(vertices):
    scaler = MinMaxScaler()
    normalized_vertices = scaler.fit_transform(vertices)
    return normalized_vertices

#function of nearest neighbour approach
def rearrange_vertices_normalized_nearest_neighbor(vertices):
    # Normalize the vertices
    normalized_vertices = normalize_values(vertices)
    
    # Find the vertex with the smallest z-value to start
    start_idx = np.argmin(normalized_vertices[:, 2])
    new_order = [start_idx]

    remaining_indices = set(range(len(normalized_vertices))) - {start_idx}

    while remaining_indices:
        last_idx = new_order[-1]
        last_vertex = normalized_vertices[last_idx]
        
        nearest_idx = min(remaining_indices, key=lambda idx: np.linalg.norm(normalized_vertices[idx] - last_vertex))
        new_order.append(nearest_idx)
        remaining_indices.remove(nearest_idx)
    
    rearranged_vertices = vertices[new_order]  # Return the original vertices in the new order
    return rearranged_vertices, new_order

#executing function of nearest neighbour on the linestrings
def rearrange_all_linestrings(linestrings):
    rearranged_linestrings = {}
    last_z_values = []  # To collect last z-values
    selected_z_values = []
    all_rearranged_vertices = {}  # To store all rearranged vertices for each line string
    for idx, line_string in linestrings.items():
        vertices = np.array(line_string.coords)
        try:
            rearranged_vertices, new_order = rearrange_vertices_normalized_nearest_neighbor(vertices)
            rearranged_linestrings[idx] = LineString(rearranged_vertices)
            last_z = get_z_value(rearranged_vertices, -1)
            first_z = get_z_value(rearranged_vertices, 0)
            last_z_values.append(last_z)
            
            if last_z > 0.75 * first_z:
                selected_z = last_z
                
            else:
                selected_z = first_z
                           
            selected_z_values.append(selected_z)
            all_rearranged_vertices[idx] = (rearranged_vertices, new_order)
                
        except Exception as e:
            print(f"Center {idx}: Rearrangement failed with error: {e}")
            rearranged_linestrings[idx] = line_string
            
    return rearranged_linestrings, last_z_values, selected_z_values, all_rearranged_vertices

#execute algorithm
linestrings_rearranged, last_z_values_Section_dir1, selected_z_values_Section_dir1, all_rearranged_vertices_Section_dir1 = rearrange_all_linestrings(linestrings)

selected_z_values = selected_z_values_Section_dir1 

#plotting of rearranged linestrings
def plot_rearranged_linestrings_plotly(linestrings, title, min_step):
    for idx, line_string in linestrings.items():
        coords = np.array(line_string.coords)
        vertex_indices = np.arange(len(coords))
        
        # Get the last z-value
        last_z_value = get_last_z_value(line_string)

        min_step = min_steps.get(idx)
        print(f"Center {idx}: min_step = {min_step}")
        
        # Check if min_step is 0, then use x and z, otherwise y and z
        if dx < dy:
            if min_step == 0:
                x_vals = coords[:, 0]  # Use x for min_step=0
                axis_title = 'X'
            else:
                x_vals = coords[:, 1]  # Use y for min_step != 0
                axis_title = 'Y'

        elif dx >= dy:
            if min_step == 90:
                x_vals = coords[:, 0]  # Use x for min_step=0
                axis_title = 'X'
            else:
                x_vals = coords[:, 1]  # Use y for min_step != 0
                axis_title = 'Y'
        
        
        fig = go.Figure()
        fig.add_trace(go.Scatter(
            x=x_vals, y=coords[:, 2], mode='markers+lines', 
            marker=dict(color=vertex_indices, colorscale='Viridis', size=8, colorbar=dict(title='Vertex Index'))
        ))
        fig.update_layout(
            title=f'{title} - Center {idx} - Last z-value: {last_z_value}', 
            xaxis_title=axis_title, 
            yaxis_title='Z',
            height=600
        )
        fig.show()


# Plot the rearranged linestrings
plot_rearranged_linestrings_plotly(linestrings_rearranged, 'Rearranged Linestrings - Section_dir1', min_step)


In [ ]:
"""second step to correct remaining potential artifacts in cross sections of 1st direction: interactive approach for manual correction"""
import os
import json
import numpy as np
from shapely.geometry import LineString
from sklearn.preprocessing import MinMaxScaler
import plotly.graph_objects as go
from dash import Dash, dcc, html, Input, Output, State, callback_context
import dash.exceptions
import matplotlib.pyplot as plt

# Load min_steps from the JSON file
with open('min_steps.json', 'r') as f:
    min_steps = json.load(f)
# Ensure min_steps is a dictionary with integer keys
min_steps = {int(k): v for k, v in min_steps.items()}

section_type = None
center_idx = None

# Normalization and rearrangement functions
def normalize_values(vertices):
    scaler = MinMaxScaler()
    normalized_vertices = scaler.fit_transform(vertices)
    return normalized_vertices

def rearrange_vertices_normalized_nearest_neighbor(vertices):
    normalized_vertices = normalize_values(vertices)
    start_idx = np.argmin(normalized_vertices[:, 2])
    new_order = [start_idx]
    remaining_indices = set(range(len(normalized_vertices))) - {start_idx}
    while remaining_indices:
        last_idx = new_order[-1]
        last_vertex = normalized_vertices[last_idx]
        nearest_idx = min(remaining_indices, key=lambda idx: np.linalg.norm(normalized_vertices[idx] - last_vertex))
        new_order.append(nearest_idx)
        remaining_indices.remove(nearest_idx)
    rearranged_vertices = vertices[new_order]
    return rearranged_vertices, new_order

def get_z_value(vertices, index):
    return vertices[index][2]  

def rearrange_all_linestrings(linestrings):
    rearranged_linestrings = {}
    all_rearranged_vertices = {}

    for idx, line_string in linestrings.items():
        vertices = np.array(line_string.coords)
        try:
            rearranged_vertices, new_order = rearrange_vertices_normalized_nearest_neighbor(vertices)
            rearranged_linestrings[idx] = LineString(rearranged_vertices)
            all_rearranged_vertices[idx] = (rearranged_vertices, new_order)
        except Exception as e:
            print(f"Center {idx}: Rearrangement failed with error: {e}")
            rearranged_linestrings[idx] = line_string

    return rearranged_linestrings, all_rearranged_vertices

#function for figure within the interactive application
def create_figure(linestring, title, vertex_order, center_idx):
    fig = go.Figure()

    # Get coordinates from LineString
    x, y = linestring.xy
    x = list(x)  # Convert to list
    y = list(y)  # Convert to list
    coords = np.array(linestring.coords)

    # Check min_step for the current center
    min_step = min_steps.get(center_idx, 1)  

    if dx < dy:
        if min_step == 0:
            z = coords[:, 2] 
            x_vals = coords[:, 0]  # Use x for min_step=0
            axis_title = 'X'
        else:
            z = coords[:, 2] 
            x_vals = coords[:, 1]  # Use y for min_step != 0
            axis_title = 'Y'

    elif dx >= dy:
        if min_step == 90:
            z = coords[:, 2]
            x_vals = coords[:, 0]  # Use x for min_step=0
            axis_title = 'X'
        else:
            z = coords[:, 2]
            x_vals = coords[:, 1]  # Use y for min_step != 0
            axis_title = 'Y'

    # Add line trace
    fig.add_trace(go.Scatter(x=x_vals, y=z, mode='lines+markers', name='LineString'))

    # Apply viridis colormap
    cmap = plt.get_cmap('viridis')
    norm = plt.Normalize(vmin=0, vmax=(len(x) - 1))
    colors = [cmap(norm(i)) for i in range(len(x))]
    colors = [f'rgba({r*255},{g*255},{b*255},{a})' for r, g, b, a in colors]

    # Scatter plot with colormap
    fig.add_trace(go.Scatter(
        x=x_vals, y=z,
        mode='markers',
        marker=dict(color=colors, size=10),
        name='Vertices'
    ))

    fig.update_layout(title=title)
    return fig


"""Step 1: Process Linestrings"""
linestrings_rearranged, all_rearranged_vertices_Section_dir1 = rearrange_all_linestrings(linestrings)

"""this function determines, which cross sections might need manual correction --> if last_z is larger than 0.9 * first_z for a section, this section
is available in the drop-down menu in the application. 0.9 is an empirically determined value to save some time potentially needed to click through 
all sections. If you would like to check all sections for the need of manual correction, set that number to > 1"""
def check_correction_needed(linestrings_rearranged, all_rearranged_vertices, section_type):
    last_z_values = []
    selected_z_values = []
    correction_flag = False
    correction_indices = []

    for idx, (rearranged_vertices, new_order) in all_rearranged_vertices.items():
        last_z = get_z_value(rearranged_vertices, -1)
        first_z = get_z_value(rearranged_vertices, 0)
        last_z_values.append(last_z)

        if last_z > 0.9 * first_z:
            correction_flag = True
            correction_indices.append(idx)
        else:
            selected_z = last_z
            selected_z_values.append(selected_z)

    if correction_flag:
        print(f"Manual correction required for {section_type}, centers: {correction_indices} before proceeding.")

    return last_z_values, selected_z_values, correction_flag, correction_indices

"""Step 2: raise correction flag and implement manual correction if needed"""
last_z_values_Section_dir1, selected_z_values_Section_dir1, flag_Section_dir1, indices_Section_dir1 = check_correction_needed(linestrings_rearranged, all_rearranged_vertices_Section_dir1, "Section_dir1")

selected_z_values = selected_z_values_Section_dir1 

if not (flag_Section_dir1):         
    print("No manual correction required. Continue processing...")

# Initialize Dash app for interactive corrections
app = Dash(__name__)

# Generate dropdown options based on correction flag
dropdown_options = []
for idx in indices_Section_dir1:
    dropdown_options.append({'label': f'Section_dir1 - Center {idx}', 'value': f'Section_dir1-{idx}'})
    
# Definition of App layout
app.layout = html.Div([
    dcc.Dropdown(
        id='direction-center-selector',
        options=dropdown_options,
        placeholder='Select a direction-center combination'
    ),
    html.Div(id='plots-container', children=[]),
    dcc.Store(id='vertex-order', data=[]),
    dcc.Store(id='vertex-order-history', data=[]),
    dcc.Store(id='current-figure-index', data=0),
    dcc.Store(id='selected-vertices', data=[]),
    dcc.Store(id='correction-info', data={'section_type': None, 'center_idx': None}),
    html.Div(id='debug', style={'display': 'none'}),
    dcc.Store(id='save-message', data=''),
    dcc.Graph(id='plot'),
    html.Button('Save', id='save-btn', n_clicks=0),
    html.Button('Undo', id='undo-btn', n_clicks=0),
    html.Div(id='save-status')
])

@app.callback(
    [Output('plot', 'figure'),
     Output('vertex-order', 'data'),
     Output('selected-vertices', 'data'),
     Output('current-figure-index', 'data'),
     Output('vertex-order-history', 'data'),
     Output('save-message', 'data')],
    [Input('save-btn', 'n_clicks'),
     Input('plot', 'clickData'),
     Input('direction-center-selector', 'value'),
     Input('undo-btn', 'n_clicks')],
    [State('vertex-order', 'data'),
     State('vertex-order-history', 'data'),
     State('current-figure-index', 'data'),
     State('selected-vertices', 'data'),
     State('correction-info', 'data')]
)

#Definition of App functionality
def update_figure(n_clicks_save, clickData, selected_direction_center, n_clicks_undo, vertex_order, vertex_order_history, current_figure_index, selected_vertices, correction_info):
    ctx = callback_context
    if not ctx.triggered:
        raise dash.exceptions.PreventUpdate

    trigger = ctx.triggered[0]['prop_id'].split('.')[0]
    save_message = ''
    section_type, center_idx = None, None

    if selected_direction_center:
        section_type, center_idx = selected_direction_center.split('-')
        center_idx = int(center_idx)

    if trigger == 'direction-center-selector' and selected_direction_center:
        if section_type == 'Section_dir1':
            linestring = list(linestrings_rearranged.values())[center_idx - 1]
        else:
            linestring = list(perpendicular_linestrings_rearranged.values())[center_idx - 1]

        vertex_order = list(range(len(linestring.coords)))
        vertex_order_history = []
        selected_vertices = []
        return create_figure(linestring, f'Linestrings 1st direction - Center {center_idx}', vertex_order, center_idx), vertex_order, selected_vertices, None, vertex_order_history, ''

    if trigger == 'plot' and clickData:
        point_index = clickData['points'][0]['pointIndex']
        if point_index >= len(vertex_order):
            raise dash.exceptions.PreventUpdate  # Prevent out-of-range index
        if len(selected_vertices) == 0:
            selected_vertices = [point_index]
        elif len(selected_vertices) == 1:
            selected_vertices.append(point_index)
            first, second = selected_vertices
            if first != second:
                vertex_order_history.append(vertex_order.copy())
                try:
                    first_idx = vertex_order[first]
                    second_idx = vertex_order[second]
                    vertex_order.remove(second_idx)
                    insert_index = vertex_order.index(first_idx) + 1
                    vertex_order.insert(insert_index, second_idx)
                except IndexError as e:
                    print(f"IndexError: {e}")
            selected_vertices = []

        if section_type == 'Section_dir1':
            coords = np.array(list(linestrings_rearranged.values())[center_idx - 1].coords)
        else:
            coords = np.array(list(perpendicular_linestrings_rearranged.values())[center_idx - 1].coords)

        rearranged_coords = coords[vertex_order]
        new_fig = create_figure(LineString(rearranged_coords), f'Linestrings 1st direction - Center {center_idx}', vertex_order, center_idx)
        return new_fig, vertex_order, selected_vertices, current_figure_index, vertex_order_history, ''

    if trigger == 'save-btn':
        if selected_vertices and len(selected_vertices) == 2:
            idx1, idx2 = selected_vertices
            try:
                vertex_order[idx1], vertex_order[idx2] = vertex_order[idx2], vertex_order[idx1]
                vertex_order_history.append(vertex_order.copy())
                selected_vertices = []
            except IndexError as e:
                print(f"IndexError: {e}")

        if section_type == 'Section_dir1':
            coords = np.array(list(linestrings_rearranged.values())[center_idx - 1].coords)
        else:
            coords = np.array(list(perpendicular_linestrings_rearranged.values())[center_idx - 1].coords)

        rearranged_coords = coords[vertex_order]
        new_fig = create_figure(LineString(rearranged_coords), f'Linestrings 1st direction - Center {center_idx}', vertex_order, center_idx)

        save_message = f"Corrected vertex order saved to corrected_vertices_direction_{section_type}_center_{center_idx}.npy"
        try:
            base_save_dir = r"C:\Users\carl\Desktop\Paper 3\random datasets standard models\Juliusburg_Koestorf-Rosenthal\Zwischenschritte" #newly set vertex order is saved to directory
            dataset_name = "10 sections 0 boreholes_hull_GE_150_smoothed"
            dataset_save_dir = os.path.join(base_save_dir, dataset_name)
            if not os.path.exists(dataset_save_dir):
                os.makedirs(dataset_save_dir)
            
            save_path = os.path.join(dataset_save_dir, f"corrected_vertices_direction_{section_type}_center_{center_idx}.npy")
            np.save(save_path, vertex_order)
            save_message = f"Corrected vertex order saved to {save_path}"
        except Exception as e:
            save_message = f"Error occurred while saving: {str(e)}"
        
        print(save_message)

        return new_fig, vertex_order, selected_vertices, current_figure_index, vertex_order_history, save_message

    if trigger == 'undo-btn' and vertex_order_history:
        vertex_order = vertex_order_history.pop()
        if section_type == 'Section_dir1':
            coords = np.array(list(linestrings_rearranged.values())[center_idx - 1].coords)
        else:
            coords = np.array(list(perpendicular_linestrings_rearranged.values())[center_idx - 1].coords)

        rearranged_coords = coords[vertex_order]
        new_fig = create_figure(LineString(rearranged_coords), f'Rearranged Linestrings - Center {center_idx}', vertex_order, center_idx)
        return new_fig, vertex_order, selected_vertices, current_figure_index, vertex_order_history, ''

    raise dash.exceptions.PreventUpdate

if __name__ == '__main__':
    app.run_server(debug=True)


In [ ]:
"""Creation of orthogonal cross sections (perpendicular to the secondary horizontal main axis)"""
import os
import pyvista as pv
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.transform import Rotation as R
from sklearn.preprocessing import MinMaxScaler
import matplotlib.cm as cm
from shapely.geometry import LineString
import json
import gc

%load_ext memory_profiler
%memit

#retrieve saved center_points
with open('center_points.json', 'r') as f:
    center_points = json.load(f)
# Convert keys by extracting numbers from string keys and converting values to NumPy arrays
center_points = {
    int(k.split('_')[-1]): np.array(v) 
    for k, v in center_points.items() 
    if int(k.split('_')[-1]) != 0  # Skip the entry with index 0
}

# Load rotated normals
with open('rotated_normals.json', 'r') as f:
    rotated_normals = json.load(f)
rotated_normals = {
    int(k.split('_')[-1]): np.array(v)  # Extract the number from the key
    for k, v in rotated_normals.items()
}
 
# Load min_step_dict and min_area_dict from JSON files
try:
    with open('min_steps.json', 'r') as f:
        min_steps = json.load(f)
    with open('min_area_dict.json', 'r') as f:
        min_area_dict = json.load(f)

    # Convert keys to integers if they are stored as strings in the JSON files
    min_steps = {int(k): v for k, v in min_steps.items()}   
    min_area_dict = {int(k): v for k, v in min_area_dict.items()}
except FileNotFoundError as e:
    print(f"Error loading dictionaries: {e}")
    min_step_dict = {}
    min_area_dict = {}



# Directory containing the manually corrected linestrings
save_dir = r"C:\Users\carl\Desktop\Paper 3\random datasets standard models\Juliusburg_Koestorf-Rosenthal\Zwischenschritte\10 sections 0 boreholes_hull_GE_150_smoothed"

# Check if the directory exists
if os.path.exists(save_dir):
    corrected_files = os.listdir(save_dir)
else:
    corrected_files = []

combined_linestrings = {}

# Function to compute the area using the shoelace formula
def shoelace_area(vertices):
    n = len(vertices)
    area = 0.5 * abs(sum(vertices[i][0]*vertices[(i+1) % n][1] - vertices[i][1]*vertices[(i+1) % n][0] for i in range(n)))
    return area

# Normalize the vertices
def normalize_values(vertices):
    scaler = MinMaxScaler()
    normalized_vertices = scaler.fit_transform(vertices)
    return normalized_vertices

def ensure_continuous_loop(points):
    if points[0] != points[-1]:
        points.append(points[0])
    return points

def ensure_continuous_loop(points):
    # Check if the first point is not equal to the last point
    if not np.array_equal(points[0], points[-1]):
        # If they are not equal, append the first point to the end to close the loop
        points = np.vstack([points, points[0]])
    return points

# Function to load the saved corrected vertex order
def load_corrected_vertex_order(file_path):
    try:
        if os.path.exists(file_path):
            return np.load(file_path)
        else:
            raise FileNotFoundError(f"File '{file_path}' not found.")
    except Exception as e:
        return None

# Function to combine manually corrected linestrings and automatically corrected linestrings into "combined_linestrings"
def combine_vertex_order_with_coords(linestring, center_index, section_type):
    section_name = f"corrected_vertices_direction_{section_type.replace(' ', '_')}_center_{center_index}.npy"
    file_path = os.path.join(save_dir, section_name)

    corrected_vertex_order = load_corrected_vertex_order(file_path)

    if corrected_vertex_order is not None:
        try:
            coords = np.array(linestring.coords)
            corrected_coords = coords[corrected_vertex_order]
            return LineString(corrected_coords)
        except Exception as e:
            print(f"Error combining vertex order with coordinates: {e}")
            return linestring
    else:
        return linestring

# Update linestrings with corrected data using the new function
for key, linestring in linestrings_rearranged.items():
    combined_linestrings[key] = combine_vertex_order_with_coords(linestring, key, "Section dir1")


"""the following 2 functions subdivide sections of 1st direction vertically into 32 intervals & determine the x & y coordinates of these lines""" 
#Function to compute y-range and evenly spaced lines for 32 subdivisions
def compute_y_range_and_lines(ys, num_lines=32):
    y_min, y_max = min(ys), max(ys)
    y_lines = np.linspace(y_min, y_max, num_lines)  # Equally spaced y-values for lines
    return y_lines

# Function to compute x-range and evenly spaced lines for 32 subdivisions
def compute_x_range_and_lines(xs, num_lines=32):
    x_min, x_max = min(xs), max(xs)
    x_lines = np.linspace(x_min, x_max, num_lines)  # Equally spaced x-values for lines
    return x_lines


#plotting the vertical lines
def plot_with_vertical_y_lines(ys, zs, title, y_lines, zmin_zmax_for_section):
    plt.figure()
    
    # Fill the area of the shape in blue
    plt.fill(ys, zs, 'b', edgecolor='black', alpha=0.5)
    
    # Plot points
    plt.plot(ys, zs, 'o')
    
    # Add vertical lines
    for idx, y_line in enumerate(y_lines):
        plt.axvline(x=y_line, color='yellow', linestyle='--', label=f'Line {idx + 1}' if idx < len(y_lines) - 1 else None)

    # Customize plot appearance
    plt.title(title)
    plt.xlabel('Y coordinates')
    plt.ylabel('Z coordinates')
    plt.grid(True)
    
    # Show plot
    plt.show()

# Function to plot the 2D XZ projection with vertical lines
def plot_with_vertical_x_lines(xs, zs, title, x_lines, zmin_zmax_for_section):
    plt.figure()
    
    # Fill the area of the shape in blue
    plt.fill(xs, zs, 'b', edgecolor='black', alpha=0.5)
    
    # Plot points
    plt.plot(xs, zs, 'o')
    
    # Add vertical lines
    for idx, x_line in enumerate(x_lines):
        plt.axvline(x=x_line, color='red', linestyle='--', label=f'Line {idx + 1}' if idx < len(x_lines) - 1 else None)

    # Customize plot appearance
    plt.title(title)
    plt.xlabel('X coordinates')
    plt.ylabel('Z coordinates')
    plt.grid(True)
    
    # Show plot
    plt.show()



"""following 2 functions determine the z values of the lines, where the lines intersect the polygon edge (only one of the functions is run per cross 
section of the 1st direction, depending on its orientation"""
def find_zmin_zmax_for_y_line(points, y_value):
    min_z_value = float('inf')
    max_z_value = -float('inf')
    
    for i in range(len(points)):
        _, y1, z1 = points[i]  # Unpack x, y, and z (ignore x)
        _, y2, z2 = points[(i + 1) % len(points)]  # Unpack next point (ignore x)
        
        # Check if the y_value intersects with the edge (y1, y2)
        if min(y1, y2) <= y_value <= max(y1, y2):
            # Calculate the z-value at the intersection of the y_line and the edge
            z_intersection = z1 + (y_value - y1) * (z2 - z1) / (y2 - y1)
            # Update min and max z values
            min_z_value = min(min_z_value, z_intersection)
            max_z_value = max(max_z_value, z_intersection)
    
    # Handle case with no intersections found
    if min_z_value == float('inf') or max_z_value == -float('inf'):
        print(f"No intersections found for y_value: {y_value} with points: {points}")
        return None, None

    return min_z_value, max_z_value


def find_zmin_zmax_for_x_line(points, x_value):
    min_z_value = float('inf')
    max_z_value = -float('inf')
    
    for i in range(len(points)):
        x1, _, z1 = points[i]  # Unpack x, y, and z (ignore y)
        x2, _, z2 = points[(i + 1) % len(points)]  # Unpack next point (ignore y)
        
        # Check if the x_value intersects with the edge (x1, x2)
        if min(x1, x2) <= x_value <= max(x1, x2):
            # Calculate the z-value at the intersection of the x_line and the edge
            z_intersection = z1 + (x_value - x1) * (z2 - z1) / (x2 - x1)
            # Update min and max z values
            min_z_value = min(min_z_value, z_intersection)
            max_z_value = max(max_z_value, z_intersection)
    
    # Handle case with no intersections found
    if min_z_value == float('inf') or max_z_value == -float('inf'):
        print(f"No intersections found for x_value: {x_value} with points: {points}")
        return None, None

    return min_z_value, max_z_value


# Initialize the plotter
plotter = pv.Plotter()

# Create a colormap for different centers
num_centers = len(center_array)
colormap = cm.get_cmap('viridis', num_centers)

# Dictionaries to store data for each section
x_lines_dict = {} 
y_lines_dict = {}
zmin_zmax_dict = {}


for idx, linestring in list(combined_linestrings.items()):
    if idx == 31:
        continue
    points = list(linestring.coords)
    # Calculate area
    sorted_points = ensure_continuous_loop(points)
    area = shoelace_area(sorted_points)

        
    xs, ys, zs = zip(*sorted_points)  # Create lists of x and y values

    y_lines = compute_y_range_and_lines(ys, 32)  # 32 vertical lines
    min_step = min_steps.get(idx, None)

    """in some cases, the index order of the lines has to be reversed depending on their orientation, so that the creation of trapezoidal segments works 
    (see below)"""
    if dx < dy:
        if min_step is not None and (0 <= min_step <= 90 or min_step in [185, 190]):
            y_lines = y_lines[::-1]

    elif dx >= dy:
            y_lines = y_lines
    
    y_lines_dict[idx] = y_lines

    x_lines = compute_x_range_and_lines(xs,32)  # 32 vertical lines
    min_step = min_steps.get(idx, None)

    if dx < dy:
            x_lines = x_lines

    elif dx >= dy:
        if min_step is not None and (0 <= min_step <= 90 or min_step in [185, 190]):  
            x_lines = x_lines[::-1]

    x_lines_dict[idx] = x_lines
    """if you want optional printing, uncomment the two lines of code below"""
    #for idx, x_lines in x_lines_dict.items():
     #   print(f"Center {idx}: X lines = {x_lines}")
    

    zmin_zmax_for_section = []
    
    # Compute Zmin and Zmax for each line
    if dx < dy:
        if min_step != 0:
            for y_line in y_lines:
                zmin, zmax = find_zmin_zmax_for_y_line(sorted_points, y_line)
                zmin_zmax_for_section.append((y_line, zmin, zmax))
    
        if min_step == 0:
            for x_line in x_lines:
                zmin, zmax = find_zmin_zmax_for_x_line(sorted_points, x_line)
                
                zmin_zmax_for_section.append((x_line, zmin, zmax))

        # Store Zmin and Zmax values for this section
        zmin_zmax_dict[idx] = zmin_zmax_for_section

    elif dx >= dy:    
        if min_step != 90:
            for y_line in y_lines:
                zmin, zmax = find_zmin_zmax_for_y_line(sorted_points, y_line)
                zmin_zmax_for_section.append((y_line, zmin, zmax))
    
        if min_step == 90:
            for x_line in x_lines:
                zmin, zmax = find_zmin_zmax_for_x_line(sorted_points, x_line)
                
                zmin_zmax_for_section.append((x_line, zmin, zmax))

        # Store Zmin and Zmax values for this section
        zmin_zmax_dict[idx] = zmin_zmax_for_section
     

    

    # Access the minimal step and area values from the dictionaries using idx
    if idx in min_steps and idx in min_area_dict:
        min_step = min_steps[idx]  # Get the min_step for this section/center
        min_area = min_area_dict[idx]  # Get the min_area for this section/center
        
        # Plot the 2D YZ projections and XZ projections with vertical lines
        plot_with_vertical_y_lines(ys, zs, f'Center {idx}: Minimal Step at {min_step:.2f}°', y_lines, zmin_zmax_for_section)
        print(f"Center {idx}:")
        print(f"Y lines: {y_lines}")
        
        plot_with_vertical_x_lines(xs, zs, f'Center {idx}: Minimal Step at {min_step:.2f}°', x_lines, zmin_zmax_for_section)
        print(f"Center {idx}:")
        print(f"X lines: {x_lines}")

    #print Z values of lines
    for zmin_zmax in zmin_zmax_for_section:
        line_value = zmin_zmax[0]
        zmin = zmin_zmax[1]
        zmax = zmin_zmax[2]
        print(f"  Line {line_value}: Zmin = {zmin}, Zmax = {zmax}")


"""------------------------------------------------------2nd step CREATION OF ORTHOGONAL SECTIONS---------------------------------------------------"""
# Initialize the plotter for visualization
plotter = pv.Plotter()


# batch processing for faster computation
start_index = 0 if 0 in combined_linestrings else 1
batch_size = 100

# Get the sorted list of keys from combined_linestrings 
keys = sorted(combined_linestrings.keys())


def calculate_horizontal_extent(points):
    """
    Calculate the horizontal extent (in the XY plane) of the trapezoidal segment. Only the distances between horizontal edges (ignoring the 
    z-coordinates) are summed. For name convention of points, see below
    """
    # Extract the x and y coordinates 
    x_a, y_a = points[0][:2]  # Point 1
    x_b, y_b = points[3][:2]  # Point 4

    # Calculate the horizontal distance between points 1 and 4
    horizontal_distance = np.sqrt((x_b - x_a) ** 2 + (y_b - y_a) ** 2)

    if dx < dy:
        if y_a > y_b:  # Cross-sections are crossing, need to be subtracted, see paper discussion
            horizontal_distance *= -1
    else:  # dx >= dy
        if x_a > x_b:  # Cross-sections are crossing, need to be subtracted, see paper discussion
            horizontal_distance *= -1

    return horizontal_distance


trapezoidal_segments = {n: [] for n in range(32)}
perpendicular_section = {n: [] for n in range(32)}
horizontal_extents = {n: 0 for n in range(32)}


"""setup to create trapezoidal segments out of two lines of same index=n from two consecutive sections of 1st direction (index=i)"""
for batch_start in range(0, len(keys), batch_size):
    batch_end = min(batch_start + batch_size, len(keys))

    for i in range(batch_start, batch_end - 1):
        # Ensure the index does not exceed bounds
        if i + 1 >= len(keys):
            break

        # Use the keys list to access the linestrings
        key_a = keys[i]
        key_b = keys[i + 1]
        
        linestring_a = combined_linestrings[key_a]  # Linestring for section center i='a'
        linestring_b = combined_linestrings[key_b]  # Linestring for section center i='a+1'

        # Extract the points for both linestrings
        points_a = list(linestring_a.coords)
        points_b = list(linestring_b.coords)

        sorted_points_a = ensure_continuous_loop(points_a)
        sorted_points_b = ensure_continuous_loop(points_b)

        xs_a, ys_a, zs_a = zip(*sorted_points_a)  # Y, X and Z coordinates for center 'a'
        xs_b, ys_b, zs_b = zip(*sorted_points_b)  # Y, X and Z coordinates for center 'a+1'

        # Get the y_values, x_values and zmin_zmax for both centers
        y_lines_a = y_lines_dict.get(key_a)
        y_lines_b = y_lines_dict.get(key_b)

        x_lines_a = x_lines_dict.get(key_a)
        x_lines_b = x_lines_dict.get(key_b)
        
        zmin_zmax_a = zmin_zmax_dict.get(key_a)
        zmin_zmax_b = zmin_zmax_dict.get(key_b)


        if x_lines_a is None or x_lines_b is None or y_lines_a is None or y_lines_b is None or zmin_zmax_a is None or zmin_zmax_b is None:
            print(f"Missing data for center {key_a} or {key_b}. Skipping this pair.")
            continue

        # Loop through each index 'n' of orthogonal direction for creating vertical lines
        for n in range(32):
            # Extract fixed coordinates for center 'a'
            x_a = x_lines_a[n]                          # X-coordinate from x_lines 
            y_a = y_lines_a[n]                          # Y-coordinate from y_lines
            zmin_a, zmax_a = zmin_zmax_a[n][1], zmin_zmax_a[n][2]  # Zmin and Zmax for 'a'

            # Extract fixed coordinates for center 'a+1'
            x_b =  x_lines_b[n]                             # X-coordinate for 'a+1'
            y_b = y_lines_b[n]                              # Y-coordinate from y_lines for 'a+1'
            zmin_b, zmax_b = zmin_zmax_b[n][1], zmin_zmax_b[n][2]  # Zmin and Zmax for 'a+1'

            points = np.array([
                [x_a, y_a, zmin_a],  # Point 1
                [x_a, y_a, zmax_a],  # Point 2
                [x_b, y_b, zmax_b],  # Point 3
                [x_b, y_b, zmin_b]   # Point 4
            ])

            # Define the face (quad) of the trapezoid by specifying the indices of the points
            faces = np.array([[4, 0, 1, 2, 3]])  # '4' indicates it's a quad (4 points)

                                
            # Calculate the horizontal extent of this trapezoid
            horizontal_extent = calculate_horizontal_extent(points)
            #print(f"Horizontal extent for trapezoidal segment at n={n}: {horizontal_extent}")

            # Add the horizontal extent to the total for this orthogonal section
            horizontal_extents[n] += horizontal_extent

        
            # Create the PolyData with the points and the face
            quad = pv.PolyData(points, faces)
            
            perpendicular_section[n].append(points)

            trapezoidal_segments[n].append(points)

for n, extent in horizontal_extents.items():
    print(f"Perpendicular section {n}: horizontal extent = {extent}")

#saving calculated horizontal extents
with open('horizontal_extents.json', 'w') as json_file:
    json.dump(horizontal_extents, json_file)
print("Horizontal_extents saved to 'Horizontal_extents.json'")

# Save the trapezoidal segments to a JSON file
with open('trapezoidal_segments.json', 'w') as json_file:
    json.dump({n: [points.tolist() for points in segments] for n, segments in trapezoidal_segments.items()}, json_file)
print("Trapezoidal segments saved to 'trapezoidal_segments.json'")


if combined_linestrings:
    max_existing_key = max(combined_linestrings.keys())  # Get the max key from combined_linestrings
else:
    max_existing_key = 0  # In case combined_linestrings is empty

#create combined trapezoids for each index n, starting from the next available key (linestrings from both directions shall be in combined_linestrings)
new_key = max_existing_key + 1


"""combining the trapezoidal segments to form orthogonal cross sections"""
for n, trapezoid_points in perpendicular_section.items(): 
    if trapezoid_points:  # If there are points to create a trapezoid
        # Combine all points into a single array
        combined_points = np.vstack(trapezoid_points)

        # Get rid of duplicate points and create a mesh for the combined trapezoid
        unique_points, unique_indices = np.unique(combined_points, axis=0, return_inverse=True)
        faces = np.array([[4, unique_indices[i], unique_indices[i+1], unique_indices[i+2], unique_indices[i+3]] 
                          for i in range(0, len(unique_indices), 4)])  # Forming quads

        # Create Pyvista PolyData for the combined trapezoid
        combined_quad = pv.PolyData(unique_points, faces)

        #define orthogonal sections you want to plot
        plot_sections = [5, 10, 15]

        if n in plot_sections:
            # Add the combined trapezoid mesh to the plotter
            plotter.add_mesh(combined_quad, color='red', opacity=0.5, label=f'Perpendicular section {n}')

        # Convert trapezoid points into a LineString
        line_coords_perpendicular = [(point[0], point[1], point[2]) for point in unique_points]  
        line_string_perpendicular = LineString(line_coords_perpendicular)

        # Append LineString to combined_linestrings with a unique key
        combined_linestrings[new_key] = line_string_perpendicular
        # Check the number of items in combined_linestrings
        new_key += 1



"""Ensuring, that sections of 1st direction and orthogonal (=perpendicular) sections get correct keys in combined_linestrings"""
Section_dir1_linestrings = {key: linestring for key, linestring in combined_linestrings.items() 
                      if 1 <= key <= 30}  
num_keys_Section_dir1 = len(Section_dir1_linestrings)
print(f"Number of keys belonging to 'Section Dir 1': {num_keys_Section_dir1}")

perpendicular_linestrings = {
    mapped_key: combined_linestrings[old_key] 
    for mapped_key, old_key in enumerate(range(32, 64))  # Map old keys 32 to 62 to new keys 0 to 20
    if old_key in combined_linestrings
}


"""plotting of defined orthogonal sections in 3D"""
# Set plot labels, grid, background, and camera position
plotter.add_mesh(mesh, opacity=0.3, color='white')
plotter.add_axes()
plotter.set_background('white')
plotter.show_grid(color='black')
plotter.view_isometric()

# Set the camera position to view along the Z-axis
plotter.view_vector([1, 0, 1])

# Show the plot
plotter.show()

# Clear memory after plotting
gc.collect()

In [ ]:
"""Step 1 of artifact correction in orthogonal cross sections: normalized nearest neighbor approach to automatically correct artifacts. 
For explanations on functions, see nearest neighbour cell for sections of 1st direction"""
import numpy as np
from shapely.geometry import LineString
from sklearn.preprocessing import MinMaxScaler
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import json

# Load min_steps from the JSON file
with open('min_steps.json', 'r') as f:
    min_steps = json.load(f)
min_steps = {int(k): v for k, v in min_steps.items()}

def get_last_z_value(line_string):
    coords = np.array(line_string.coords)
    return coords[-1, 2]  

def get_z_value(vertices, index):
    return vertices[index][2]  

def normalize_values(vertices):
    scaler = MinMaxScaler()
    normalized_vertices = scaler.fit_transform(vertices)
    return normalized_vertices
    
def rearrange_vertices_normalized_nearest_neighbor(vertices):
    # Normalize the vertices
    normalized_vertices = normalize_values(vertices)
    
    # Find the vertex with the smallest z-value to start
    start_idx = np.argmin(normalized_vertices[:, 2])
    new_order = [start_idx]

    remaining_indices = set(range(len(normalized_vertices))) - {start_idx}

    while remaining_indices:
        last_idx = new_order[-1]
        last_vertex = normalized_vertices[last_idx]
        
        nearest_idx = min(remaining_indices, key=lambda idx: np.linalg.norm(normalized_vertices[idx] - last_vertex))
        new_order.append(nearest_idx)
        remaining_indices.remove(nearest_idx)
    
    rearranged_vertices = vertices[new_order]  # Return the original vertices in the new order
    return rearranged_vertices, new_order

def rearrange_all_linestrings_with_comparison(linestrings):
    rearranged_linestrings = {}
    last_z_values = []  # To collect last z-values
    selected_z_values = []
    all_rearranged_vertices = {}  # To store all rearranged vertices for each line string
    for idx, line_string in linestrings.items():
        vertices = np.array(line_string.coords)
        try:
            rearranged_vertices, new_order = rearrange_vertices_normalized_nearest_neighbor(vertices)
            rearranged_linestrings[idx] = LineString(rearranged_vertices)
            last_z = get_z_value(rearranged_vertices, -1)
            first_z = get_z_value(rearranged_vertices, 0)
            last_z_values.append(last_z)
            
            if last_z > 0.75 * first_z:
                selected_z = last_z
            else:
                selected_z = first_z            
            selected_z_values.append(selected_z)
            all_rearranged_vertices[idx] = (rearranged_vertices, new_order)
                
        except Exception as e:
            print(f"Center {idx}: Rearrangement failed with error: {e}")
            rearranged_linestrings[idx] = line_string
            
    return rearranged_linestrings, last_z_values, selected_z_values, all_rearranged_vertices


# Run the rearrangement
perpendicular_linestrings_rearranged, last_z_values_perpendicular, selected_z_values_perpendicular, all_rearranged_vertices_perpendicular = rearrange_all_linestrings_with_comparison(perpendicular_linestrings)
selected_z_values = selected_z_values_perpendicular

#plotting function for rearranged linestrings
def plot_rearranged_linestrings_plotly(linestrings, title):
    for idx, line_string in linestrings.items():
        coords = np.array(line_string.coords)
        vertex_indices = np.arange(len(coords))
        
        # Get the last z-value
        last_z_value = get_last_z_value(line_string)

        fig = go.Figure()
        if dx < dy:
            # Plot in the YZ plane
            fig.add_trace(go.Scatter(
                x=coords[:, 1], y=coords[:, 2], mode='markers+lines',
                marker=dict(color=vertex_indices, colorscale='Viridis', size=8, colorbar=dict(title='Vertex Index'))
            ))
            fig.update_layout(
                title=f'{title} - Center {idx}', 
                xaxis_title='Y', 
                yaxis_title='Z',
                height=600
            )

        elif dx >= dy:
            # Plot in the XZ plane
            fig.add_trace(go.Scatter(
                x=coords[:, 0], y=coords[:, 2], mode='markers+lines',
                marker=dict(color=vertex_indices, colorscale='Viridis', size=8, colorbar=dict(title='Vertex Index'))
            ))
            fig.update_layout(
                title=f'{title} - Center {idx}', 
                xaxis_title='X', 
                yaxis_title='Z',
                height=600
            )

        # Show the figure
        fig.show()


# Plot the rearranged linestrings
plot_rearranged_linestrings_plotly(perpendicular_linestrings_rearranged, 'Rearranged Linestrings - Perpendicular_Section')


In [ ]:
"""interactive approach orthogonal sections: : manually correct potentially remaining artefacts
Check the cell of the interactive approach for sections of 1st direction for additional comments"""
import os
import numpy as np
from shapely.geometry import LineString
from sklearn.preprocessing import MinMaxScaler
import plotly.graph_objects as go
from dash import Dash, dcc, html, Input, Output, State, callback_context
import dash.exceptions
import matplotlib.pyplot as plt
import json

# Load min_steps from the JSON file
with open('min_steps.json', 'r') as f:
    min_steps = json.load(f)
# Ensure min_steps is a dictionary with integer keys
min_steps = {int(k): v for k, v in min_steps.items()}

section_type = None
center_idx = None

# Normalization and rearrangement functions
def normalize_values(vertices):
    scaler = MinMaxScaler()
    normalized_vertices = scaler.fit_transform(vertices)
    return normalized_vertices

def rearrange_vertices_normalized_nearest_neighbor(vertices):
    normalized_vertices = normalize_values(vertices)
    start_idx = np.argmin(normalized_vertices[:, 2])
    new_order = [start_idx]
    remaining_indices = set(range(len(normalized_vertices))) - {start_idx}
    while remaining_indices:
        last_idx = new_order[-1]
        last_vertex = normalized_vertices[last_idx]
        nearest_idx = min(remaining_indices, key=lambda idx: np.linalg.norm(normalized_vertices[idx] - last_vertex))
        new_order.append(nearest_idx)
        remaining_indices.remove(nearest_idx)
    rearranged_vertices = vertices[new_order]
    return rearranged_vertices, new_order

def get_z_value(vertices, index):
    return vertices[index][2]  

def rearrange_all_linestrings(linestrings):
    rearranged_linestrings = {}
    all_rearranged_vertices = {}

    for idx, line_string in linestrings.items():
        vertices = np.array(line_string.coords)
        try:
            rearranged_vertices, new_order = rearrange_vertices_normalized_nearest_neighbor(vertices)
            rearranged_linestrings[idx] = LineString(rearranged_vertices)
            all_rearranged_vertices[idx] = (rearranged_vertices, new_order)
        except Exception as e:
            print(f"Center {idx}: Rearrangement failed with error: {e}")
            rearranged_linestrings[idx] = line_string

    return rearranged_linestrings, all_rearranged_vertices

def create_figure(linestring, title, vertex_order):
    fig = go.Figure()
    
    coords = np.array(linestring.coords)
    
    if dx < dy:
        # Plot in the YZ plane
        x = list(coords[:, 1])  # Y values
        y = list(coords[:, 2])  # Z values
        fig.add_trace(go.Scatter(
            x=x, y=y, mode='lines+markers', name='LineString'
        ))
        fig.update_layout(
            xaxis_title='Y', 
            yaxis_title='Z'
        )
    else:
        # Plot in the XZ plane
        x = list(coords[:, 0])  # X values
        y = list(coords[:, 2])  # Z values
        fig.add_trace(go.Scatter(
            x=x, y=y, mode='lines+markers', name='LineString'
        ))
        fig.update_layout(
            xaxis_title='X', 
            yaxis_title='Z'
        )
    
    # Apply viridis colormap
    cmap = plt.get_cmap('viridis')
    norm = plt.Normalize(vmin=0, vmax=(len(x) - 1))
    colors = [cmap(norm(i)) for i in range(len(x))]
    colors = [f'rgba({r*255},{g*255},{b*255},{a})' for r, g, b, a in colors]

    # Scatter plot with colormap
    fig.add_trace(go.Scatter(
        x=x, y=y,
        mode='markers',
        marker=dict(color=colors, size=10),
        name='Vertices'
    ))

    fig.update_layout(title=title, height=600)
    return fig

#apply rearrangement
perpendicular_linestrings_rearranged, all_rearranged_vertices_perpendicular = rearrange_all_linestrings(perpendicular_linestrings)

def check_correction_needed(linestrings_rearranged, all_rearranged_vertices, section_type):
    last_z_values = []
    selected_z_values = []
    correction_flag = False
    correction_indices = []

    for idx, (rearranged_vertices, new_order) in all_rearranged_vertices.items():
        last_z = get_z_value(rearranged_vertices, -1)
        first_z = get_z_value(rearranged_vertices, 0)
        last_z_values.append(last_z)

        if last_z > 1 * first_z: # recommended to check all orthogonal sections for potential artifacts
            correction_flag = True
            correction_indices.append(idx)
        else:
            selected_z = last_z
            selected_z_values.append(selected_z)

    if correction_flag:
        print(f"Manual correction needed for {section_type}, centers: {correction_indices} bf proceeding.")

    return last_z_values, selected_z_values, correction_flag, correction_indices

# Step 2: Raise Flag and Implement Manual Correction if Needed
last_z_values_perpendicular, selected_z_values_perpendicular, flag_perpendicular, indices_perpendicular = check_correction_needed(perpendicular_linestrings_rearranged, all_rearranged_vertices_perpendicular, "Perpendicular Direction")
selected_z_values =  selected_z_values_perpendicular

if not (flag_perpendicular):
    print("No manual correction required. Continue processing...")

# Initialize Dash app for interactive corrections
app = Dash(__name__)

# Generate dropdown options based on correction flag
dropdown_options = []
for idx in indices_perpendicular:
    dropdown_options.append({'label': f'Perpendicular_Direction- Center {idx}', 'value': f'Perpendicular_Direction-{idx}'})

app.layout = html.Div([
    dcc.Dropdown(
        id='direction-center-selector',
        options=dropdown_options,
        placeholder='Select a direction-center combination'
    ),
    html.Div(id='plots-container', children=[]),
    dcc.Store(id='vertex-order', data=[]),
    dcc.Store(id='vertex-order-history', data=[]),
    dcc.Store(id='current-figure-index', data=0),
    dcc.Store(id='selected-vertices', data=[]),
    dcc.Store(id='correction-info', data={'section_type': None, 'center_idx': None}),
    html.Div(id='debug', style={'display': 'none'}),
    dcc.Store(id='save-message', data=''),
    dcc.Graph(id='plot'),
    html.Button('Save', id='save-btn', n_clicks=0),
    html.Button('Undo', id='undo-btn', n_clicks=0),
    html.Div(id='save-status')
])

@app.callback(
    [Output('plot', 'figure'),
     Output('vertex-order', 'data'),
     Output('selected-vertices', 'data'),
     Output('current-figure-index', 'data'),
     Output('vertex-order-history', 'data'),
     Output('save-message', 'data')],
    [Input('save-btn', 'n_clicks'),
     Input('plot', 'clickData'),
     Input('direction-center-selector', 'value'),
     Input('undo-btn', 'n_clicks')],
    [State('vertex-order', 'data'),
     State('vertex-order-history', 'data'),
     State('current-figure-index', 'data'),
     State('selected-vertices', 'data'),
     State('correction-info', 'data')]
)
def update_figure(n_clicks_save, clickData, selected_direction_center, n_clicks_undo, vertex_order, vertex_order_history, current_figure_index, selected_vertices, correction_info):
    ctx = callback_context
    if not ctx.triggered:
        raise dash.exceptions.PreventUpdate

    trigger = ctx.triggered[0]['prop_id'].split('.')[0]
    save_message = ''
    section_type, center_idx = None, None

    if selected_direction_center:
        section_type, center_idx = selected_direction_center.split('-')
        center_idx = int(center_idx)

    if trigger == 'direction-center-selector' and selected_direction_center:
        if section_type == 'Section_dir1':
            linestring = list(linestrings_rearranged.values())[center_idx - 1]
        else: 
            #section_type == 'Perpendicular Direction'
            adjusted_idx = center_idx - 31
            linestring = list(perpendicular_linestrings_rearranged.values())[adjusted_idx-1]

        vertex_order = list(range(len(linestring.coords)))
        vertex_order_history = []
        selected_vertices = []
        return create_figure(linestring, f'Perpendicular Linestrings - Center {center_idx}', vertex_order), vertex_order, selected_vertices, None, vertex_order_history, ''

    if trigger == 'plot' and clickData:
        point_index = clickData['points'][0]['pointIndex']
        if point_index >= len(vertex_order):
            raise dash.exceptions.PreventUpdate  # Prevent out-of-range index
        if len(selected_vertices) == 0:
            selected_vertices = [point_index]
        elif len(selected_vertices) == 1:
            selected_vertices.append(point_index)
            first, second = selected_vertices
            if first != second:
                vertex_order_history.append(vertex_order.copy())
                try:
                    first_idx = vertex_order[first]
                    second_idx = vertex_order[second]
                    vertex_order.remove(second_idx)
                    insert_index = vertex_order.index(first_idx) + 1
                    vertex_order.insert(insert_index, second_idx)
                except IndexError as e:
                    print(f"IndexError: {e}")
            selected_vertices = []

        if section_type == 'Section_dir1':
            coords = np.array(list(linestrings_rearranged.values())[center_idx - 1].coords)
        else: 
            #section_type == 'Perpendicular Direction'
            adjusted_idx = center_idx - 31
            coords = np.array(list(perpendicular_linestrings_rearranged.values())[adjusted_idx-1].coords)

        rearranged_coords = coords[vertex_order]
        new_fig = create_figure(LineString(rearranged_coords), f'Perpendicular Linestrings - Center {center_idx}', vertex_order)
        return new_fig, vertex_order, selected_vertices, current_figure_index, vertex_order_history, ''

    if trigger == 'save-btn':
        if selected_vertices and len(selected_vertices) == 2:
            idx1, idx2 = selected_vertices
            try:
                vertex_order[idx1], vertex_order[idx2] = vertex_order[idx2], vertex_order[idx1]
                vertex_order_history.append(vertex_order.copy())
                selected_vertices = []
            except IndexError as e:
                print(f"IndexError: {e}")

        if section_type == 'Section_dir1':
            coords = np.array(list(linestrings_rearranged.values())[center_idx - 1].coords)
        else: 
            #section_type == 'Perpendicular Direction'
            adjusted_idx = center_idx - 31
            coords = np.array(list(perpendicular_linestrings_rearranged.values())[adjusted_idx-1].coords)

        rearranged_coords = coords[vertex_order]
        new_fig = create_figure(LineString(rearranged_coords), f'Perpendicular Linestrings - Center {center_idx}', vertex_order)

        save_message = f"Corrected vertex order saved to corrected_vertices_direction_{section_type}_center_{center_idx}.npy"
        try:
            base_save_dir = r"C:\Users\carl\Desktop\Paper 3\random datasets standard models\Juliusburg_Koestorf-Rosenthal\Zwischenschritte" #newly set vertex order is saved to directory again
            dataset_name = "10 sections 0 boreholes_hull_GE_150_smoothed"
            dataset_save_dir = os.path.join(base_save_dir, dataset_name)
            if not os.path.exists(dataset_save_dir):
                os.makedirs(dataset_save_dir)
            
            save_path = os.path.join(dataset_save_dir, f"corrected_vertices_direction_{section_type}_center_{center_idx}.npy")
            np.save(save_path, vertex_order)
            save_message = f"Corrected vertex order saved to {save_path}"
        except Exception as e:
            save_message = f"Error occurred while saving: {str(e)}"
        
        print(save_message)

        return new_fig, vertex_order, selected_vertices, current_figure_index, vertex_order_history, save_message

    if trigger == 'undo-btn' and vertex_order_history:
        vertex_order = vertex_order_history.pop()
        if section_type == 'Section_dir1':
            coords = np.array(list(linestrings_rearranged.values())[center_idx - 1].coords)
        else: 
            #section_type == 'Perpendicular Direction'
            adjusted_idx = center_idx - 31
            coords = np.array(list(perpendicular_linestrings_rearranged.values())[adjusted_idx-1].coords)

        rearranged_coords = coords[vertex_order]
        new_fig = create_figure(LineString(rearranged_coords), f'Perpendicular Linestrings - Center {center_idx}', vertex_order)
        return new_fig, vertex_order, selected_vertices, current_figure_index, vertex_order_history, ''

    raise dash.exceptions.PreventUpdate

if __name__ == '__main__':
    app.run_server(debug=True)


In [ ]:
"""combination of manually corrected sections with automatically corrected sections"""
import os
import numpy as np
from shapely.geometry import LineString
import matplotlib.pyplot as plt

# Directory containing the corrected linestrings
save_dir = r"C:\Users\carl\Desktop\Paper 3\random datasets standard models\Juliusburg_Koestorf-Rosenthal\Zwischenschritte\10 sections 0 boreholes_hull_GE_150_smoothed"

# Check if the directory exists
if os.path.exists(save_dir):
    corrected_files = os.listdir(save_dir)
else:
    corrected_files = []

# Combine unchanged and corrected linestrings
combined_linestrings = {}

# Function to load the saved corrected vertex order
def load_corrected_vertex_order(file_path):
    try:
        if os.path.exists(file_path):
            return np.load(file_path)
        else:
            raise FileNotFoundError(f"File '{file_path}' not found.")
    except Exception as e:
        return None

# Function to combine vertex order with coordinates
def combine_vertex_order_with_coords(linestring, center_index, section_type):
    section_name = f"corrected_vertices_direction_{section_type.replace(' ', '_')}_center_{center_index}.npy"
    file_path = os.path.join(save_dir, section_name)

    corrected_vertex_order = load_corrected_vertex_order(file_path)

    if corrected_vertex_order is not None:
        try:
            coords = np.array(linestring.coords)
            corrected_coords = coords[corrected_vertex_order]
            return LineString(corrected_coords)
        except Exception as e:
            print(f"Error combining vertex order with coordinates: {e}")
            return linestring
    else:
        return linestring

# Integrate corrections for linestrings_rearranged
for key, linestring in linestrings_rearranged.items():
    combined_linestrings[int(key)] = combine_vertex_order_with_coords(linestring, int(key), "Section dir1")

# Print the current state of combined_linestrings
print(f"Keys in combined_linestrings after Section Dir 1: {list(combined_linestrings.keys())}")

# Now we need to generate unique keys for the perpendicular linestrings
max_existing_key = max(combined_linestrings.keys()) if combined_linestrings else 0

# Integrate corrections for perpendicular_linestrings_rearranged
for key, linestring in perpendicular_linestrings_rearranged.items():
    try:
        key_as_int = int(key)  # Ensure the key is an integer
        new_key = max_existing_key + 1  # Start from the next available key
        
        # Find an available key that does not conflict
        while new_key in combined_linestrings:
            new_key += 1

        combined_linestrings[new_key] = combine_vertex_order_with_coords(linestring, key_as_int, "Perpendicular_Direction")
        max_existing_key = new_key  # Update max_existing_key for future use
    except ValueError:
        print(f"Key '{key}' could not be converted to an integer.")
    except Exception as e:
        print(f"An error occurred while processing key '{key}': {e}")

# Print the final state of combined_linestrings
print(f"Final number of keys in combined_linestrings: {len(combined_linestrings)}")
print(f"Keys in combined_linestrings: {list(combined_linestrings.keys())}")


# Function to create a simple figure with corrected headings
def create_simple_figure(linestring, slice_type, center_index):
    coords = np.array(linestring.coords)  # Assuming linestring.coords gives you an array of [x, y, z]
    
    if "Slice 1" in slice_type:
        title = f"Slice 1, Center {center_index}"
        #x = coords[:, 1]  # Y-values
     #   y = coords[:, 2]  # Z-values

        if dx is not None and dy is not None:  # Ensure dx and dy are passed in
            if dx > dy:
                # Plot in the YZ plane
                x = coords[:, 1]  # Y-values (second column)
                y = coords[:, 2]  # Z-values (third column)
            else:
                # Plot in the XZ plane
                x = coords[:, 0]  # X-values (first column)
                y = coords[:, 2]  # Z-values (third column)
        else:
            # Fallback to YZ plane if dx and dy are not provided
            x = coords[:, 1]  # Y-values
            y = coords[:, 2]  # Z-values

    
    elif "Perpendicular Slice" in slice_type:
        title = f"Perpendicular Slice, Center {center_index}"

        # Apply conditional logic for plotting either YZ or XZ plane
        if dx is not None and dy is not None:  # Ensure dx and dy are passed in
            if dx < dy:
                # Plot in the YZ plane
                x = coords[:, 1]  # Y-values (second column)
                y = coords[:, 2]  # Z-values (third column)
            else:
                # Plot in the XZ plane
                x = coords[:, 0]  # X-values (first column)
                y = coords[:, 2]  # Z-values (third column)
        else:
            # Fallback to YZ plane if dx and dy are not provided
            x = coords[:, 1]  # Y-values
            y = coords[:, 2]  # Z-values
    else:
        # Default case for unknown or other slice types
        title = f"Combined Linestrings - Center {center_index}"
        x = coords[:, 1]  # Y-values
        y = coords[:, 2]  # Z-values

    # Convert to lists for plotting
    x = list(x)
    y = list(y)

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=x, y=y, mode='lines+markers', name='LineString'))

    # Apply viridis colormap
    cmap = plt.get_cmap('viridis')
    norm = plt.Normalize(vmin=0, vmax=len(x) - 1)
    colors = [cmap(norm(i)) for i in range(len(x))]
    colors = [f'rgba({r*255},{g*255},{b*255},{a})' for r, g, b, a in colors]

    # Scatter plot with colormap
    fig.add_trace(go.Scatter(
        x=x, y=y,
        mode='markers',
        marker=dict(color=colors, size=10),
        name='Vertices'
    ))

    fig.update_layout(title=title)
    return fig
    
# Visualize combined linestrings with corrected headings
for idx, linestring in combined_linestrings.items():
    if idx in linestrings_rearranged:
        slice_type = "Slice 1"
        center_index = idx  # Center index is the same as the key for Slice 1
    elif idx > len(linestrings_rearranged):
        slice_type = "Perpendicular Slice"
        center_index = idx - len(linestrings_rearranged)-1  # Center index starts from 0 for Perpendicular Slice
    else:
        slice_type = "Unknown"
        center_index = idx  # Default case

    # Create and show the figure
    fig = create_simple_figure(linestring, slice_type, center_index)
    fig.show()



In [ ]:
"""optional: visualization of all corrected cross section of both directions in top view"""
import matplotlib.pyplot as plt
import numpy as np
import pyvista as pv

# Initialize the plot
fig = plt.figure(figsize=(12, 12))
ax  = fig.gca()
# Create a colormap with 31 distinct colors
colors = plt.cm.tab20c(np.linspace(0, 1, 31))

# Specify the index to exclude


# Initialize variables to store data ranges
y_min, y_max = float('inf'), float('-inf')
x_min, x_max = float('inf'), float('-inf')

# Loop through combined_linestrings to plot all except the excluded index
for idx, linestring in combined_linestrings.items():
    # Check if the current index matches the excluded index
    #if idx == excluded_index:
     #   continue

    # Determine the slice type and center index
    if idx > len(linestrings_rearranged):
        slice_type = "Perpendicular Slice"
        center_index = idx - len(linestrings_rearranged) - 1  # Center index for Perpendicular Slices

        # Extract coordinates from the linestring
        coords = np.array(linestring.coords)  # Convert to NumPy array
        x_coords = coords[:, 0]  # X-coordinates
        y_coords = coords[:, 1]  # Y-coordinates

        # Update the data range
        y_min, y_max = min(y_min, y_coords.min()), max(y_max, y_coords.max())
        x_min, x_max = min(x_min, x_coords.min()), max(x_max, x_coords.max())

        # Plot the Perpendicular Slice in X-Y plane
        plt.plot(
            y_coords, x_coords,
            marker='o',
            color=colors[center_index % len(colors)],
            linestyle='--',
            label=f'Orthogonal section {center_index}'
        )

# Customize the plot
plt.title(f'Orthogonal sections in X-Y Plane', fontsize=16)
plt.xlabel('Y', fontsize=14)
plt.ylabel('X', fontsize=14)
plt.grid(True)
plt.axis('equal')  # Ensure equal scaling on both axes

# Adjust the limits based on the data ranges with a small margin
margin = 0.05  # 5% margin
plt.xlim(y_min - margin * (y_max - y_min), y_max + margin * (y_max - y_min))
plt.ylim(x_min - margin * (x_max - x_min), x_max + margin * (x_max - x_min))
ax.tick_params(axis='both', which='major', labelsize=20)
# Invert the X-axis
plt.gca().invert_yaxis()

# Add legend
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

# Show the plot
plt.show()

In [ ]:
"""Dimensional Measurements: measurement of the data in the horizontal and vertical directions between the intersections of the measurement 
lines with the linestring polygons"""
import os
import numpy as np
from shapely.geometry import LineString
import pandas as pd
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import json

with open('center_points.json', 'r') as f:
    center_points = json.load(f)
center_points = {
    int(k.split('_')[-1]): np.array(v) 
    for k, v in center_points.items() 
    if int(k.split('_')[-1]) != 0  # Skip the entry with index 0
}
with open('rotated_norms.json', 'r') as f:
     rotated_norms = json.load(f)
rotated_norms = {
    int(k.split('_')[-1]): np.array(v)  # Extract the number from the key
    for k, v in rotated_norms.items()
}
with open('horizontal_extents.json', 'r') as f:
    horizontal_extents = json.load(f)
horizontal_extents = {int(k): v for k, v in horizontal_extents.items()}

try:
    with open('min_steps.json', 'r') as f:
        min_steps = json.load(f)
    with open('min_area_dict.json', 'r') as f:
        min_area_dict = json.load(f)
    # Convert keys to integers if they are stored as strings in the JSON files
    min_steps = {int(k): v for k, v in min_steps.items()}   
    min_area_dict = {int(k): v for k, v in min_area_dict.items()}
except FileNotFoundError as e:
    print(f"Error loading dictionaries: {e}")
    min_step_dict = {}
    min_area_dict = {}


def sort_points_by_angle(points, centroid):
    def angle(p):
        return np.arctan2(p[1] - centroid[1], p[0] - centroid[0])
    return sorted(points, key=angle)

# Normalize the vertices
def normalize_values(vertices):
    scaler = MinMaxScaler()
    normalized_vertices = scaler.fit_transform(vertices)
    return normalized_vertices

#function to rotate and project cross section parallel to the YZ plane
def rotate_to_x_axis(vertices, normal):
    normal = normal / np.linalg.norm(normal)  # Normalize the normal vector
    target = np.array([1, 0, 0])  # Target vector along the x-axis

    # Ensure normal is not aligned with the target vector to avoid zero rotation
    if np.allclose(normal, target) or np.allclose(normal, -target):
        return vertices, np.eye(3)  # No rotation needed if already aligned

    # Calculate the rotation needed to align the normal with the x-axis
    rotation_vector = np.cross(normal, target)
    rotation_vector /= np.linalg.norm(rotation_vector)  # Normalize the rotation vector
    angle = np.arccos(np.clip(np.dot(normal, target), -1.0, 1.0))
    
    K = np.array([
        [0, -rotation_vector[2], rotation_vector[1]],
        [rotation_vector[2], 0, -rotation_vector[0]],
        [-rotation_vector[1], rotation_vector[0], 0]
    ])
    rotation_matrix = np.eye(3) + np.sin(angle) * K + (1 - np.cos(angle)) * np.dot(K, K)

    if np.linalg.det(rotation_matrix) < 0:
        rotation_matrix[:, 1] = -rotation_matrix[:, 1]
        
    # Apply the rotation to the vertices
    rotated_vertices = vertices @ rotation_matrix.T 
    return rotated_vertices, rotation_matrix



all_linestrings = combined_linestrings

def determine_section_type(key, num_linestrings):

    try:
        key = int(key)  # Ensure key is an integer
    except ValueError as e:
        print(f"Error converting key to int: {key}. Error: {e}")
        return None  # Return None if conversion fails

    if key < num_linestrings:
        return "Section_dir1"
    else:
        return "Perpendicular_Direction"

    print(f"Section type for key {key}: {section_type}")
    
    return section_type



def rescale_linestring_coords(linestring, new_min, new_max, dx, dy):
    # Extract the current coordinates of the linestring
    points = np.array(linestring.coords)  # Convert to numpy array for easier manipulation
    
    # Determine if we should work on x or y coordinates based on dx and dy
    if dx >= dy:
        current_min = np.min(points[:, 0])  # Extract x-values (1st column)
        current_max = np.max(points[:, 0])  # X-range
    else:
        current_min = np.min(points[:, 1])  # Extract y-values (2nd column)
        current_max = np.max(points[:, 1])  # Y-range
    
    # Calculate the scaling factor based on the current and new ranges
    scale = (new_max - new_min) / (current_max - current_min)
    
    # Rescale the coordinates
    new_points = []
    for x, y, z in points:
        if dx >= dy:
            new_x = new_min + (x - current_min) * scale  # Scale the x-value
            new_points.append((new_x, y, z))  # Keep y and z unchanged
        else:
            new_y = new_min + (y - current_min) * scale  # Scale the y-value
            new_points.append((x, new_y, z))  # Keep x and z unchanged
    
    # Return a new linestring with the rescaled coordinates
    return LineString(new_points)


def create_rescaling_figure(linestring, section_type, center_index, dx, dy):
    # Set the title based on section_type and center_index
    if "Section_dir1" in section_type:
        title = f"Direction 1, center {center_index}"
    elif "Perpendicular_Direction" in section_type:
        title = f"Perpendicular_Direction, center {center_index}"
    else:
        title = f"Combined Linestrings - Center {center_index}"

    fig = go.Figure()

    coords = np.array(linestring.coords)  
    
    if dx >= dy:
        y = coords[:, 0]  # Use X-values (1st column)
    else:
        y = coords[:, 1]  # Use Y-values (2nd column)
    
    z = coords[:, 2]  # Z-values (3rd column)

    # Convert to lists for plotting
    y = list(y)
    z = list(z)

    # Add line trace for the linestring
    fig.add_trace(go.Scatter(x=y, y=z, mode='lines+markers', name='LineString'))

    # Apply viridis colormap
    cmap = plt.get_cmap('viridis')
    norm = plt.Normalize(vmin=0, vmax=len(y) - 1)
    colors = [cmap(norm(i)) for i in range(len(y))]
    colors = [f'rgba({r*255},{g*255},{b*255},{a})' for r, g, b, a in colors]

    # Add a scatter plot for the vertices with the colormap
    fig.add_trace(go.Scatter(
        x=y, y=z,
        mode='markers',
        marker=dict(color=colors, size=10),
        name='Vertices'
    ))
    x_label = 'X' if dx >= dy else 'Y'
    fig.update_layout(title=title, xaxis_title='Y', yaxis_title='Z')
    return fig

# Define max_Section_dir1_key 
max_Section_dir1_key = max(linestrings_rearranged)  

# Loop through all_linestrings to determine orthogonal (=perpendicular) sections
for idx, (key, linestring) in enumerate(all_linestrings.items()):
    if key in linestrings_rearranged:
        section_type = "Section_dir1"
        center_index = key  # For Direction 1, the center index is the key itself
    elif key > max_Section_dir1_key:  # Keys greater than max_Section_dir1_key belong to Perpendicular sections
        section_type = "Perpendicular_Direction"
        center_index = key - (max_Section_dir1_key + 1)  
    else:
        section_type = "Unknown"
        center_index = key  # Fallback for unknown section types

    # Rescale Perpendicular Sections
    if section_type == "Perpendicular_Direction":  
        # Retrieve the corresponding horizontal extent for this center index
        horizontal_extent = horizontal_extents.get(center_index, None)
        
        if horizontal_extent is not None:
            # Check the condition between dx and dy to rescale either x or y
            if dx >= dy:
                # Rescale the x-values to the range [0, horizontal_extent]
                xmin_section = 0
                xmax_section = horizontal_extent
                rescaled_linestring = rescale_linestring_coords(linestring, xmin_section, xmax_section, dx, dy)
            else:
                # Rescale the y-values to the range [0, horizontal_extent]
                ymin_section = 0
                ymax_section = horizontal_extent
                rescaled_linestring = rescale_linestring_coords(linestring, ymin_section, ymax_section, dx, dy)
            
            # **Save the rescaled linestring back into all_linestrings**
            all_linestrings[key] = rescaled_linestring
       
    else:
        # For "Direction 1", keep the linestring unchanged in all_linestrings
        all_linestrings[key] = linestring


def ensure_continuous_loop_yz(points):
    # Check if the first point is not equal to the last point
    if not np.array_equal(points[0], points[-1]):
        # If they are not equal, append the first point to the end to close the loop
        points = np.vstack([points, points[0]])
    return points

def ensure_continuous_loop(points):
    """
    Ensure that the polygon forms a closed loop.
    """
    first_point = points[0]
    last_point = points[-1]
    
    # Check if the first and last points are the same
    if first_point != last_point:
        points.append(first_point)    
    return points


# Calculate the shoelace area
def shoelace_area(points):
    n = len(points)
    area = 0.0
    for i in range(n):
        j = (i + 1) % n
        area += points[i][0] * points[j][1]
        area -= points[j][0] * points[i][1]
    return abs(area) / 2.0

# Measure polygon line length along horizontal axis at given z value
def measure_polygon_line_length_y(points, y_value):
    min_z_value = float('inf')
    max_z_value = -float('inf')
    
    for i in range(len(points)):
        z1, y1 = points[i]
        z2, y2 = points[(i + 1) % len(points)]
        
        if min(y1, y2) <= y_value <= max(y1, y2):
            z_intersection = z1 + (y_value - y1) * (z2 - z1) / (y2 - y1)
            min_z_value = min(min_z_value, z_intersection)
            max_z_value = max(max_z_value, z_intersection)
                
    return min_z_value, max_z_value

# Measure polygon line length along z at given y value
def measure_polygon_line_length_z(points, z_value):
    min_y_value = float('inf')
    max_y_value = -float('inf')
    
    for i in range(len(points)):
        z1, y1 = points[i]
        z2, y2 = points[(i + 1) % len(points)]
        
        if min(z1, z2) <= z_value <= max(z1, z2):
            y_intersection = y1 + (z_value - z1) * (y2 - y1) / (z2 - z1)
            min_y_value = min(min_y_value, y_intersection)
            max_y_value = max(max_y_value, y_intersection)
                
    return min_y_value, max_y_value



#functions ensuring to create correct title of linestring
def create_title(section_type, center_index):
    if "Section_dir1" in section_type:
        return f"Direction 1, center {center_index}"
    elif "Perpendicular_Direction" in section_type:
        return f"Perpendicular_Direction, center {center_index}"
    else:
        return f"Combined Linestrings - Center {center_index}"  # Default title if not recognized

num_linestrings = len(combined_linestrings)
center_index = key
# Convert to integer if necessary
if isinstance(key, str) and key.startswith('rotated_n_'):
    # Extract the numeric part of the key
    key = int(key.split('_')[-1])
else:
    key = int(key)  # Ensure key is an integer


section_type = determine_section_type(key, num_linestrings)
title = create_title(section_type, key)


# Sanitize the title for use in file names
def sanitize_filename(filename):
    # Remove characters not allowed in filenames
    forbidden_chars = '<>:"/\\|?*'
    return "".join(c for c in filename if c not in forbidden_chars).replace(" ", "_")

#function to create the figures of the linestrings of 1st direction
def create_simple_figure(y_values, z_values, title, area):
    fig = go.Figure()

    if y_values[0] != y_values[-1] or z_values[0] != z_values[-1]:
        y_values.append(y_values[0])
        z_values.append(z_values[0])


    fig.add_trace(go.Scatter(
        y=z_values, x=y_values,
        mode='lines+markers',
        name='LineString',
        line=dict(color='blue')
    ))

    # Apply viridis colormap
    cmap = plt.get_cmap('viridis')
    norm = plt.Normalize(vmin=0, vmax=len(y_values) - 1)
    colors = [cmap(norm(i)) for i in range(len(y_values))]
    colors = [f'rgba({int(r*255)},{int(g*255)},{int(b*255)},{a})' for r, g, b, a in colors]

    # Scatter plot with colormap
    fig.add_trace(go.Scatter(
        y=z_values, x=y_values,
        mode='markers',
        marker=dict(color=colors, size=10),
        name='Vertices'
    ))

    fig.update_layout(
        title=f'{title} (Area: {area:.2f})',
        xaxis_title='Y',
        yaxis_title='Z',
        showlegend=True,
        autosize=True,
        width=800,  
        height=800  
    )
    
    # Set isotropic axis
    fig.update_layout(
        xaxis=dict(scaleanchor='y', constrain='domain'),
        yaxis=dict(constrain='domain')
    )

    return fig

#function to create the figures of the linestrings of orthogonal direction
def create_simple_figure_perp(linestring, title, area):
    fig = go.Figure()

    coords = np.array(linestring.coords)  

    if dx < dy:
            # Plot in the YZ plane
        y = list(coords[:, 1])  # Y-values (second column)
        z = list(coords[:, 2])  # Z-values (third column)
    else:
            # Plot in the XZ plane
        y = list(coords[:, 0])  # X-values (first column)
        z = list(coords[:, 2])  # Z-values (third column)    
    

    if (y[0], z[0]) != (y[-1], z[-1]):
        y.append(y[0])
        z.append(z[0])

    # Convert to lists for plotting
    y = list(y)
    z = list(z)

    fig.add_trace(go.Scatter(x=y, y=z, mode='lines+markers', name='LineString'))

    # Apply viridis colormap
    cmap = plt.get_cmap('viridis')
    norm = plt.Normalize(vmin=0, vmax=(len(y)) - 1)
    colors = [cmap(norm(i)) for i in range(len(y))]
    colors = [f'rgba({r*255},{g*255},{b*255},{a})' for r, g, b, a in colors]

    # Scatter plot with colormap
    fig.add_trace(go.Scatter(
        x=y, y=z,
        mode='markers',
        marker=dict(color=colors, size=10),
        name='Vertices'
    ))

    fig.update_layout(title=f'{title} (Area: {area:.2f})')
    return fig

#function defining how measurements and plotting is conducted for sections of 1st direction
def calculate_and_plot_results_Section_dir1(yz_projections,  dataset_name):
    base_dir = r"C:\Users\carl\Desktop\Paper 3\random datasets standard models\Juliusburg_Koestorf-Rosenthal\Structural dimensions\Raw data"
    dataset_dir = os.path.join(base_dir, dataset_name)

    # Create dataset directory if it does not exist
    if not os.path.exists(dataset_dir):
        os.makedirs(dataset_dir)

    # Create subdirectories for 1st and 2nd direction
    first_direction_dir = os.path.join(dataset_dir, '1st direction')
    second_direction_dir = os.path.join(dataset_dir, '2nd direction')

    if not os.path.exists(first_direction_dir):
        os.makedirs(first_direction_dir)

    if not os.path.exists(second_direction_dir):
        os.makedirs(second_direction_dir)

    for idx, sorted_points in yz_projections.items():

        ys, zs = zip(*sorted_points)  # Extract Y and Z values
        section_type = "Section_dir1"  
        center_index = idx 

        # Skip marginal sections (index 0 & 31) to avoid artifacts in dimensional measurements --> ending up with 20 sections
        if (section_type == "Section_dir1" and center_index in [0, 31]) or (section_type == "Perpendicular_Direction" and center_index in [0, 31]):
            continue
        
        # Generate title
        title = create_title(section_type, center_index)
    
        area = shoelace_area(sorted_points)
        print(f"Calculated area for line {idx}: {area}")
        
        # Extract y and z values
        z_values = [point[1] for point in sorted_points]
        y_values = [point[0] for point in sorted_points]

        fig = create_simple_figure(ys, zs,  title, area)

        ymin_section = np.min(y_values)
        ymax_section = np.max(y_values)
        zmin_section = np.min(z_values)
        zmax_section = np.max(z_values)

        # Definition of number of measurements per dimension
        num_intervals = 10

        # Calculating the interval length
        dy_section = (ymax_section - ymin_section) / (num_intervals + 1)
        dz_section = (zmax_section - zmin_section) / (num_intervals + 1)
        
        # Measuring lengths and plot the results
        y_values_section = np.linspace(ymin_section + dy_section, ymax_section - dy_section, num_intervals)
        z_values_section = np.linspace(zmin_section + dz_section, zmax_section - dz_section, num_intervals)

        # Add vertical and horizontal lines
        for y_section in y_values_section:
            fig.add_trace(go.Scatter(
                x=[y_section, y_section],
                y=[zmin_section, zmax_section],
                mode='lines',
                line=dict(color='red', width=2, dash='dash'),
                name=f'Y-section {y_section:.2f}'
            ))
        
        for z_section in z_values_section:
            fig.add_trace(go.Scatter(
                x=[ymin_section, ymax_section],
                y=[z_section, z_section],
                mode='lines',
                line=dict(color='green', width=2, dash='dash'),
                name=f'Z-section {z_section:.2f}'
            ))

        fig.show()
    
        # Measure polygon line lengths
        z_lengths = []
        y_lengths = []
    
        print("\nY-Range at specified Z-values:")
        for z_val in z_values_section:
            extent = measure_polygon_line_length_y(sorted_points, z_val)
            length = extent[1] - extent[0]
            y_lengths.append(length)
            print(f"z = {z_val}: y-range = {extent[0]} to {extent[1]}")
            print(f"z = {z_val}: horizontal length = {length}")
        
        print("\nZ-Range at specified Y-values:")
        for y_val in y_values_section:
            extent = measure_polygon_line_length_z(sorted_points, y_val)
            length = extent[1] - extent[0]
            z_lengths.append(length)
            print(f"y = {y_val}: z-range = {extent[0]} to {extent[1]}")
            print(f"y = {y_val}: vertical length = {length}")
        
        # Create DataFrame to save results
        results_df = pd.DataFrame({
            'Z Value': z_values_section,
            'Horizontal Length': y_lengths,
            'Y Value': y_values_section,
            'Vertical Length': z_lengths
        })

        results_df['Area'] = [area] + [None] * (len(results_df) - 1)
    
        # Determine filename and path based on the section type
        filename_dim = f'{sanitize_filename(title)}_dimensions.csv'
        if section_type == "Section_dir1":
            file_path_dim = os.path.join(first_direction_dir, filename_dim)
        elif section_type == "Perpendicular_Direction":
            file_path_dim = os.path.join(second_direction_dir, filename_dim)
        else:
            continue
    
        # Save results to CSV
        results_df.to_csv(file_path_dim, index=False)
    
        print(f"Dimensions saved to '{file_path_dim}'")

"""function defining how measurements and plotting is conducted for sections of orthogonal direction"""
def calculate_and_plot_results_Perpendicular_Direction(all_linestrings,  dataset_name):
    base_dir = r"C:\Users\carl\Desktop\Paper 3\random datasets standard models\Juliusburg_Koestorf-Rosenthal\Structural dimensions\Raw data"
    dataset_dir = os.path.join(base_dir, dataset_name)

    # Create dataset directory if it does not exist
    if not os.path.exists(dataset_dir):
        os.makedirs(dataset_dir)

    # Create subdirectories for 1st and 2nd direction
    first_direction_dir = os.path.join(dataset_dir, '1st direction')
    second_direction_dir = os.path.join(dataset_dir, '2nd direction')

    if not os.path.exists(first_direction_dir):
        os.makedirs(first_direction_dir)

    if not os.path.exists(second_direction_dir):
        os.makedirs(second_direction_dir)


    for idx, linestring in all_linestrings.items():
        # Calculate the relative index for Perpendicular sections by adjusting the offset by -1
        relative_index = idx - len(linestrings_rearranged) - 1

        # Check if this relative index corresponds to a Perpendicular_Direction
        if relative_index in perpendicular_linestrings_rearranged:
            section_type = "Perpendicular_Direction"
            center_index = relative_index
        else:
            continue  # Skip if not a Perpendicular section

        # Skip marginal sections (index 0 & 31) to avoid artifacts in dimensional measurements --> ending up with 20 sections
        if center_index in [0, 31]:
            continue
        
        # Generate title
        title = create_title(section_type, center_index)
        
        points = list(linestring.coords)

        if dx < dy:
            yz_points = [(y, z) for (x, y, z) in points]
        
            # Calculate area
            sorted_points = ensure_continuous_loop(yz_points)
            area = shoelace_area(sorted_points)
            
            # Extract y and z values
            ys, zs = zip(*sorted_points)  # Create lists of y and z values
            y_values = [point[0] for point in sorted_points]
            z_values = [point[1] for point in sorted_points]
        
        else:
            xz_points = [(x, z) for (x, y, z) in points]
        
            # Calculate area
            sorted_points = ensure_continuous_loop(xz_points)
            area = shoelace_area(sorted_points)
            
            # Extract x and z values
            xs, zs = zip(*sorted_points)  # Create lists of x and z values
            y_values = [point[0] for point in sorted_points]  
            z_values = [point[1] for point in sorted_points]  

        fig = create_simple_figure_perp(linestring, title, area)
   
        ymin_section = np.min(y_values)
        ymax_section = np.max(y_values)
        zmin_section = np.min(z_values)
        zmax_section = np.max(z_values)

        #Definition of number of measurements per dimension
        num_intervals = 10

        # Calculate the interval length
        dy_section = (ymax_section - ymin_section) / (num_intervals + 1)
        dz_section = (zmax_section - zmin_section) / (num_intervals + 1)
        
        # Measure lengths and plot the results
        y_values_section = np.linspace(ymin_section + dy_section, ymax_section - dy_section, num_intervals)
        z_values_section = np.linspace(zmin_section + dz_section, zmax_section - dz_section, num_intervals)

         # Add vertical and horizontal lines
        for y_section in y_values_section:
            fig.add_trace(go.Scatter(
                x=[y_section, y_section],
                y=[zmin_section, zmax_section],
                mode='lines',
                line=dict(color='red', width=2, dash='dash'),
                name=f'Y-section {y_section:.2f}'
            ))
        
        for z_section in z_values_section:
            fig.add_trace(go.Scatter(
                x=[ymin_section, ymax_section],
                y=[z_section, z_section],
                mode='lines',
                line=dict(color='green', width=2, dash='dash'),
                name=f'Z-section {z_section:.2f}'
            ))

        fig.show() 
    
        # Measure polygon line lengths
        z_lengths = []
        y_lengths = []
    
        print("\nY-Range at specified Z-values:")
        for z_val in z_values_section:   
            extent = measure_polygon_line_length_y(sorted_points, z_val)
            length = extent[1] - extent[0]
            y_lengths.append(length)
            print(f"z = {z_val}: y-range = {extent[0]} to {extent[1]}")
            print(f"z = {z_val}: horizontal length = {length}")
        
        print("\nZ-Range at specified Y-values:")
        for y_val in y_values_section:
            extent = measure_polygon_line_length_z(sorted_points, y_val)
            length = extent[1] - extent[0]
            z_lengths.append(length)
            print(f"y = {y_val}: z-range = {extent[0]} to {extent[1]}")
            print(f"y = {y_val}: vertical length = {length}")
        
        # Create DataFrame to save data
        results_df = pd.DataFrame({
            'Z Value': z_values_section,
            'Horizontal Length': y_lengths,
            'Y Value': y_values_section,
            'Vertical Length': z_lengths
        })

        results_df['Area'] = [area] + [None] * (len(results_df) - 1)
    
        # Determine filename and path based on the section type
        filename_dim = f'{sanitize_filename(title)}_dimensions.csv'
        if section_type == "Section_dir1":
            file_path_dim = os.path.join(first_direction_dir, filename_dim)
        elif section_type == "Perpendicular_Direction":
            file_path_dim = os.path.join(second_direction_dir, filename_dim)
        else:
            continue
    
        # Save results to CSV
        results_df.to_csv(file_path_dim, index=False)
    
        print(f"Dimensions saved to '{file_path_dim}'")




"""------------------------------------------code block to set up measurements on the sections of 1st direction--------------------------------------"""
Section_dir1_linestrings = {}

for idx, linestring in all_linestrings.items():
    section_type = determine_section_type(idx, num_linestrings)  # Determine the section type
    if section_type == "Section_dir1":
        Section_dir1_linestrings[idx] = linestring  # Only keep linestrings of type Direction 1

num_Section_dir1_linestrings = len(Section_dir1_linestrings)

yz_projections = {}

for idx, linestring in Section_dir1_linestrings.items():
    try:
        center = center_points[idx]  # Get the unique center point for the current idx
    except KeyError:
       
        continue  # Skip this iteration if the center point does not exist

    key = int(idx)  # Ensure key is an integer (if idx is not guaranteed to be one)
    
    # Access rotated normals using integer key directly
    if key in rotated_norms:  
        rotated_n = rotated_norms[key]  # Get the rotated normal vector for this section
    else:
        continue  

    # Get points from the linestring
    points = np.array(linestring.coords)  # Extract points as NumPy array

    # Ensure the dimensionality of points and center matches
    if points.shape[1] != center.shape[0]:
        print(f"Dimensionality mismatch: Points shape {points.shape}, Center shape {center.shape} for idx {idx}")
        continue  # Skip this linestring if there's a dimensionality mismatch

    # Step 1: Centering the points
    vertices_centered = points - center
    
    # Step 2: Rotate the vertices to align normal of section with the X-axis
    rotated_vertices, _ = rotate_to_x_axis(vertices_centered, rotated_n)
    rotated_vertices += center
    
    # Step 3: Define YZ projection
    yz_projection = rotated_vertices[:, 1:3]  # Y on X-axis, Z on Y-axis

    # Ensure all points are sorted and form a loop
    sorted_points = ensure_continuous_loop_yz(yz_projection)  

    ys, zs = zip(*sorted_points)  

    yz_projections[idx] = sorted_points 


# run the measurement functions
calculate_and_plot_results_Section_dir1(yz_projections, "10 sections 0 boreholes_hull_GE_150_smoothed")
calculate_and_plot_results_Perpendicular_Direction(all_linestrings, "10 sections 0 boreholes_hull_GE_150_smoothed")


In [ ]:
"""Calculation of gradients and curvatures on the consective vertices of the cross sections"""

import numpy as np
import csv
import json
from shapely.geometry import LineString

# Function to compute gradients (either y and z or x and z coordinates)
def compute_gradients(coords, use_x=False):
    gradients = []
    for i in range(len(coords) - 1):
        if use_x:
            x1, z1 = coords[i][0], coords[i][2]  # Use x and z
            x2, z2 = coords[i + 1][0], coords[i + 1][2]
            gradient = (z2 - z1) / (x2 - x1) if x2 != x1 else np.inf
        else:
            y1, z1 = coords[i][1], coords[i][2]  # Use y and z
            y2, z2 = coords[i + 1][1], coords[i + 1][2]
            gradient = (z2 - z1) / (y2 - y1) if y2 != y1 else np.inf
        
        gradients.append(gradient)
    return gradients

# Function to compute curvatures (either y and z or x and z coordinates)
def compute_curvatures(coords, use_x=False):
    curvatures = []
    for i in range(1, len(coords) - 1):
        if use_x:
            x0, z0 = coords[i - 1][0], coords[i - 1][2]
            x1, z1 = coords[i][0], coords[i][2]
            x2, z2 = coords[i + 1][0], coords[i + 1][2]

            # Lengths of the sides of the triangle (2D with x and z)
            a = np.sqrt((x1 - x0)**2 + (z1 - z0)**2)
            b = np.sqrt((x2 - x1)**2 + (z2 - z1)**2)
            c = np.sqrt((x2 - x0)**2 + (z2 - z0)**2)
        else:
            y0, z0 = coords[i - 1][1], coords[i - 1][2]
            y1, z1 = coords[i][1], coords[i][2]
            y2, z2 = coords[i + 1][1], coords[i + 1][2]

            # Lengths of the sides of the triangle (2D with y and z)
            a = np.sqrt((y1 - y0)**2 + (z1 - z0)**2)
            b = np.sqrt((y2 - y1)**2 + (z2 - z1)**2)
            c = np.sqrt((y2 - y0)**2 + (z2 - z0)**2)

        # Semiperimeter
        s = (a + b + c) / 2

        # Area of the triangle using Heron's formula (2D)
        area = np.sqrt(s * (s - a) * (s - b) * (s - c))

        # Curvature (2D)
        curvature = (4 * area) / (a * b * c) if a * b * c != 0 else np.inf
        curvatures.append(curvature)
    return curvatures



# Define the path to the CSV file
csv_file_path = r"C:\Users\carl\Desktop\Paper 3\random datasets standard models\Juliusburg_Koestorf-Rosenthal\Grad & curv\Raw data\10 sections 0 boreholes_hull_GE_150_smoothed.csv"

# Load min_steps from JSON
with open('min_steps.json', 'r') as f:
    min_steps = json.load(f)
    min_steps = {int(k): v for k, v in min_steps.items()} 

# List to hold all the data
data_rows = []

max_slice_1_key = max(linestrings_rearranged)  # Assuming max_slice_1_key is derived from your "Slice 1"
for idx, (key, linestring) in enumerate(all_linestrings.items()):
    coords = np.array(linestring.coords)
    #print(f"Coordinates for slice {key}:")
    #print(coords)

    # Determine slice type and center index
    if key in linestrings_rearranged:
        slice_type = "Slice 1"
        center_index = key  # For Slice 1, the center index is the key itself
    elif key > max_slice_1_key:  # Keys greater than max_slice_1_key belong to Perpendicular Slices
        slice_type = "Perpendicular Slice"
        center_index = key - (max_slice_1_key + 1)  # Adjust the index to start from 0 for Perpendicular Slices
    else:
        slice_type = "Unknown"
        center_index = key  # Fallback for unknown slice types
    
    # Skip specific slice centers
    if center_index in [0, 31]: 
        continue
    
    # Check if it's a Perpendicular Slice, then choose whether to use x or y
    use_x = False  # Default: use y-coordinates
    

##########################################################################################################################################
    if slice_type == "Slice 1" and key in min_steps:
        min_step = min_steps[key]
        if dx < dy and min_step == 0:
            use_x = True  # Use x-z if dx < dy and min_step is 0
        elif dx >= dy and min_step == 90:
            use_x = True  # Use x-z if dx >= dy and min_step is 90

    # Implement logic for Perpendicular Slice
    elif slice_type == "Perpendicular Slice":
        if dx >= dy:
            use_x = True  # Use x-coordinates if dx >= dy
    
    # Compute gradients (either using x and z or y and z)
    gradients = compute_gradients(coords, use_x=use_x)
    print(f"{slice_type} Center {center_index} Gradients ({'x-z' if use_x else 'y-z'}): {gradients}")
    #print(f"Slice {slice_type} Center {center_index} coordinates: {coords}")

    print()
"""curvature data ended up not being used in the final workflow"""    
    # Compute curvatures (either using x and z or y and z)
    curvatures = compute_curvatures(coords, use_x=use_x)
    print(f"{slice_type} Center {center_index} Curvatures ({'x-z' if use_x else 'y-z'}): {curvatures}")
    print()
    
    # Collect data
    for gradient, curvature in zip(gradients, curvatures):
        data_rows.append([f"{slice_type} Center {center_index}", gradient, curvature])

# Write data to CSV
with open(csv_file_path, mode='w', newline='') as file:
    writer = csv.writer(file)
    
    # Write header
    writer.writerow(["Description", "Gradients", "Curvatures"])
    
    # Write data rows
    writer.writerows(data_rows)

print(f"Data saved to {csv_file_path}")

